In [1]:
import os
import re
import subprocess
import sys
import time
from pathlib import Path

# --- CC-IN2P3 / Huma-Num: served locally via Ollama, NOT raw PyTorch/transformers. ---
# `module load ollama` + `ollama list` should already show this pulled:

OLLAMA_MODEL = "qwen3.5:35b"
# OLLAMA_MODEL = "qwen2.5:7b"
# OLLAMA_MODEL = "qwen3:30b"

_HOST_PORT_RE = re.compile(r"^[\w\.\-]+:\d+$")

def _resolve_ollama_host() -> str:
    """Don't hardcode the port. `module load ollama` on this cluster sets its
    own OLLAMA_HOST (your terminal showed it binding to 127.0.0.1:65330, not
    the default 11434 -- almost certainly to avoid port clashes between users
    sharing a login/compute node). We look, in order, for:
      1. OLLAMA_HOST already in this kernel's environment
      2. OLLAMA_HOST as set by `module load ollama` in a fresh login shell
         (Jupyter kernels often don't inherit modulefile env vars otherwise)
      3. the plain default, as a last resort

    NOTE: `bash -lc` sources the full login profile, which on this cluster
    prints a "Currently Loaded Modulefiles" banner to stdout unconditionally
    (before our own `echo` even runs). We can't suppress that banner from
    here, so instead we only trust the LAST non-empty line of stdout, and
    validate it actually looks like host:port before accepting it.
    """
    val = os.environ.get("OLLAMA_HOST")
    source = "kernel environment"
    if not val:
        try:
            raw = subprocess.run(
                ["bash", "-lc", "module load ollama >/dev/null 2>&1; echo $OLLAMA_HOST"],
                capture_output=True, text=True, timeout=30,
            ).stdout
            lines = [ln.strip() for ln in raw.splitlines() if ln.strip()]
            candidate = lines[-1] if lines else ""
            if _HOST_PORT_RE.match(candidate):
                val = candidate
                source = "module load ollama (login shell)"
            elif candidate:
                print(f"NOTE: module-load probe printed extra banner text; last line "
                      f"({candidate!r}) doesn't look like host:port, ignoring it.")
        except Exception:
            pass
    if not val:
        val = "127.0.0.1:11434"
        source = "default fallback -- likely wrong on this cluster, see warning below"
    # Ollama's OLLAMA_HOST convention is host:port with no scheme; our HTTP
    # client needs a scheme.
    if not val.startswith("http://") and not val.startswith("https://"):
        val = f"http://{val}"
    print(f"OLLAMA_HOST resolved to {val} (source: {source})")
    if source.startswith("default"):
        print("WARNING: could not find an OLLAMA_HOST set by the module system. "
              "Run `echo $OLLAMA_HOST` in the SAME terminal where `ollama list` "
              "works, and hardcode that value here if this keeps failing.")
    return val

OLLAMA_HOST = _resolve_ollama_host()

# Ollama defaults num_ctx to 2048 tokens no matter what the model supports --
# far too small for this prompt. The system prompt (with the full relevance
# scope, topic taxonomy, and few-shot calibration examples) is now roughly
# 4000-5000 tokens on its own; with a long document (MAX_CHARS=20000 chars,
# worst case close to 20000 tokens for dense CJK text) plus generation
# headroom, 8192 was silently at risk of truncating input or output -- which
# would look exactly like the earlier degenerate-output bugs. qwen3:30b
# natively supports much larger context, so there's no reason to run this
# tight; set generously.
OLLAMA_NUM_CTX = 32768

# Qwen3 (unlike Qwen2.5) auto-enables an invisible "thinking" reasoning pass
# in Ollama unless explicitly told not to -- burning part of every
# num_predict budget on a <think> block before the actual JSON answer, with
# no benefit for this structured-extraction task. Keep this False for real
# runs; set True only to inspect the model's reasoning for debugging a
# specific document (its content shows up in segments_debug_<variant>.jsonl
# as "thinking_preview").
OLLAMA_THINK = False

# Opt-in second pass: for every SINGLE_SEGMENT/MULTIPLE_SEGMENTS result, ask
# the model to re-check its own segment boundaries, whether it missed any
# other relevant block, and whether it padded in an irrelevant one --
# roughly doubles inference cost for those documents (WHOLE_DOCUMENT and
# NO_RELEVANT_SEGMENT results are never re-verified). Worth it if boundary/
# completeness/purity errors are still frequent in your samples; leave off
# for routine full-corpus runs where the extra cost isn't justified.
VERIFY_SEGMENTATION = False

LOCAL_INPUT_CSV = "./data/shenbao_clean.csv"
LOCAL_OUTPUT_DIR = Path("./data")

# For multi-run/full batches, keep this False so resume-by-doc_id works
# across notebook restarts (the JSONL is append-only and doc_ids already
# present are skipped -- see load_done_ids() in the worker script).
CLEAN_LOCAL_OUTPUT_DIR = False

BATCH_SIZE = 1000  # process up to this many NEW (undone) docs per run (None = all remaining)

# Which system-prompt variant to run: "full" is the long, heavily-annotated
# prompt this notebook has accumulated across many rounds of fixes; "trimmed"
# is a deliberately condensed "highest-signal core" (~56% shorter) built to
# test whether a shorter prompt is actually more reliable, rather than
# diluting the model's attention with redundant restatement. Output files
# are suffixed with the variant name (segments_full.csv vs
# segments_trimmed.csv) so both can be run and compared side by side --
# see the comparison cell below. Existing progress from before this option
# existed is auto-migrated into the "full" variant's files on first run.
PROMPT_VARIANT = "full"
# PROMPT_VARIANT = "trimmed"

def _variant_paths(variant: str):
    d = LOCAL_OUTPUT_DIR
    return {
        "jsonl": d / f"segments_{variant}.jsonl",
        "csv": d / f"segments_{variant}.csv",
        "debug": d / f"segments_debug_{variant}.jsonl",
        "log": d / f"run_{variant}.log",
    }

# A full corpus run (tens of thousands of docs at a few seconds each) can
# easily take multiple days. RUN_IN_BACKGROUND=True detaches the worker
# process from this Jupyter kernel (start_new_session + logging to a file)
# so it keeps running even if the browser tab closes, the kernel restarts,
# or the JupyterHub session idles out -- use the log-tailing cell to watch
# progress without blocking. Keep this False only for quick foreground
# smoke tests with a small BATCH_SIZE.
RUN_IN_BACKGROUND = True
LOG_PATH = _variant_paths(PROMPT_VARIANT)["log"]

MAX_TOKENS = 6144       # generation cap per call (per-attempt; retries get progressively more)
TEMPERATURE = 0.0
MAX_CHARS = 20000       # input truncation cap, in characters (see smart_truncate in the worker script)
CHECKPOINT_EVERY = 10
MAX_DEGENERATE_RETRIES = 1

WORKER_SCRIPT_PATH = Path("./segment_articles.py")





SEGMENT_ARTICLES_PY = r'''


#!/usr/bin/env python3
from __future__ import annotations

import argparse
import csv
import json
import os
import re
import time
from pathlib import Path
from typing import Any, Dict

import pandas as pd

# ---- Segmentation label space ----
SEGMENTATION_LABELS = {"WHOLE_DOCUMENT", "SINGLE_SEGMENT", "MULTIPLE_SEGMENTS", "NO_RELEVANT_SEGMENT"}

# ---- Relevance label space ----
# NOTE: since a segment only exists when the segmentation step already judged it
# relevant, this label's only job is US vs non-US, not relevant vs not-relevant.
# Enforced fully deterministically in repair() via relevance_with_us_backstop() /
# mentions_us() rather than trusted from the model -- this set just documents
# the two valid values for readers of this file.
RELEVANCE_LABELS = {"RELEVANT_US", "RELEVANT_NON_US"}

# ---- Topic taxonomy (fixed, closed set; replaces free-text topic_type) ----
# Defined with the research team so the model can no longer invent new
# categories or vary the wording of an existing one.
TOPIC_LABELS = {"policy", "debate", "biography", "organization", "other"}

TOPIC_LABEL_DESCRIPTIONS = """- policy: regulations, scholarships/stipends, state or party funding (including 庚款/Boxer Indemnity used for educational, academic, or cultural purposes), STATE/party policy on sending people abroad, selecting personnel for overseas training, sending party members to study abroad, or on their return, examinations for returned/outgoing students, visas/certificates for students, general numbers/statistics about students abroad or study-abroad trends, and STATE supervision of students abroad (e.g. a government-appointed student supervisor/inspector 監督). The defining feature is a STATE or party body/regulation/policy, or an aggregate/statistical view, as the main subject.
- organization: a specific association/group/institution acting as an ACTOR -- sender/recipient of a petition/telegram, event organizer, an association mentioned by its own NAME, a school/university (the institution itself), or a publication/editorial office tied to a student group. Includes a society's founding or activities (especially a student/alumni association, 同學會), association meetings, farewell parties, conferences, banquets, welcome receptions, mobilization campaigns, statistical surveys, or NON-STATE assistance to students. A government official or foreign commercial attache merely attending or being mentioned does NOT make it policy -- the dominant actor is what matters.
- biography: names and biographical information EXPLICITLY given about a real individual -- their education, degree, career, family, a professional role or appointment, or a specific person's OWN past/future study-abroad experience stated as a general fact. Also includes self-funded or government-sponsored students and their families, and a legal case concerning an actual, identifiable returned student (a real person's case is still about that person, not commentary). Includes collective biographies (two or more named students/alumni with their schools/degrees/achievements together).
- debate: opinions, moral arguments, questioning of policies, general discussion/critique of the study-abroad phenomenon or returned students as a category, study abroad mentioned as historical context/timeline, the returned-student community as a general category/descriptive phrase (not a specific association), or a vague/hypothetical figure ("a certain student", 某生, 某留學生) used satirically or rhetorically to actually ARGUE a moral or political point. Content set in an anti-Japanese protest or political-mobilization context involving students stays RELEVANT and is typically debate (or organization if a specific group is the actor) -- the protest framing does not remove it from scope.
- other: relevant material that does not fit any of the categories above. This includes a PURELY FICTIONAL STORY (invented characters/narrative with no real person and no actual argument being made) -- distinguish this from debate: if the piece is making a real rhetorical/moral ARGUMENT via a stand-in figure, it's debate; if it's just a story being told, it's other.

TOPIC_TYPE DISAMBIGUATION -- the SAME term (庚款/留學/留美/留學生/游美/同學會 etc.) can point to different
labels depending on what ROLE it plays in THIS segment, not just whether it's present. Match the role,
and apply multiple labels together whenever more than one genuinely applies (this is a MULTI-LABEL
field -- see the schema below):
- Term names a STATE/party policy, funding decision, examination, visa/certificate rule, or general
  statistics/trends about students abroad -> policy.
- Term names a specific association/group/institution as an ACTOR (organizing, petitioning, being
  named, being the school itself, running a publication, or a society's founding/activities) ->
  organization.
- Term describes ONE named person's own education/background/experience, a professional role or
  appointment, a self-funded/sponsored student and their family, or a legal case about a real,
  identifiable returned student -> biography.
- Term refers to the general PHENOMENON/CATEGORY of study abroad, a critique or general discussion of
  it, historical context/timeline, the returned-student community as a general category (not one
  specific association), or a stand-in figure making a real rhetorical/moral ARGUMENT -> debate.
- Term appears in a purely fictional story (invented narrative, no real argument being made) -> other,
  NOT debate.
- Term refers to a GROUP or technical staff sent abroad for training (a policy action naming specific
  people) -> policy AND biography together.
- Term refers to the appointment of a specific individual/faculty/staff who was educated/trained abroad
  to a role at a specific institution -> biography AND organization together (their personal
  appointment + the institution); a role/appointment mentioned without a specific institution acting
  is biography alone."""

# Doc-level fields: segmentation summary only.
# Exactly one explanation + one confidence; explanation is one sentence.
BASE_FIELDS = [
    "doc_id",
    "date",
    "title",
    "source",
    # segmentation
    "segmentation_label",
    "segmentation_confidence",
    "segmentation_explanation",
    # non-empty ONLY for PROCESSING_ERROR rows (infra/parse failures) -- lets
    # you filter these out of the CSV directly instead of confusing them
    # with genuine NO_RELEVANT_SEGMENT judgements.
    "processing_error",
]

# Span-level fields (one row per segment).
SEGMENT_FIELDS = [
    "segment_index",
    "segment_text",
    "segment_verbatim",
    "topic_type",
    "topic_policy",
    "topic_debate",
    "topic_biography",
    "topic_organization",
    "topic_other",
    "relevance_to_original_topic",
    "relevance_evidence",
    "gender_mentioned",
    "gender_evidence",
]

CSV_FIELDS = BASE_FIELDS + SEGMENT_FIELDS

CHECKPOINT_EVERY = int(os.environ.get("CHECKPOINT_EVERY", "10"))

# If a completed doc's segmentation_confidence == 0.0 AND its explanation is
# empty, we treat that as a degenerate/incomplete generation (not a genuine
# "no relevant material" judgement) and retry once with more room to generate.
# See PATCH_VERSION notes below for why this is not purely a token-limit fix.
MAX_DEGENERATE_RETRIES = int(os.environ.get("MAX_DEGENERATE_RETRIES", "1"))

# If this many docs IN A ROW come back as a genuine processing failure (not
# a content judgement -- see PROCESSING_ERROR / is_degenerate), pause and
# re-check Ollama's health rather than silently grinding through the rest
# of the corpus writing hundreds of error placeholders. This is exactly the
# scenario behind a mysterious run of consecutive confidence=0 rows.
CONSECUTIVE_ERROR_PAUSE_THRESHOLD = int(os.environ.get("CONSECUTIVE_ERROR_PAUSE_THRESHOLD", "10"))

# Opt-in second pass: re-examines a SINGLE_SEGMENT/MULTIPLE_SEGMENTS result
# against the full document, specifically checking segment boundary
# completeness, whether any other relevant block was missed, and whether an
# irrelevant item got padded in alongside a real one -- the three
# segmentation failure modes that kept resurfacing across many rounds of
# prompt-only fixes. Off by default since it roughly doubles the cost of
# every eligible document (one extra full inference call); WHOLE_DOCUMENT
# and NO_RELEVANT_SEGMENT results are never verified since boundary/
# completeness concerns don't apply to them the same way, and
# NO_RELEVANT_SEGMENT already has the cheaper, more targeted relevance
# recheck (see call_model_with_relevance_recheck).
VERIFY_SEGMENTATION = os.environ.get("VERIFY_SEGMENTATION", "false").strip().lower() in ("1", "true", "yes")

SYSTEM_PROMPT_FULL = f"""You are a historian's segmentation assistant for OCR/OLR historical newspaper data.

Your task is NOT to summarize.
Your task is to delineate the largest semantic unit(s) in a document that contain relevant study-abroad context, and then classify each relevant unit.

ABSOLUTE RULE -- read this first, it is the single most common mistake: if ANY part of the document
(title or body) identifies a person, group, or organization as a returned/overseas/foreign-trained
student, a study-abroad alumni/students' association, or otherwise foreign-trained personnel (using
terms like 留學生, 留美, 留日, 留英, 游學, 出洋, 同學會, 華僑學生, 庚款生, or a description of
practical/technical training abroad), the document IS relevant -- no matter what the rest of the
article actually discusses. Do NOT require the article's main topic to literally BE "study abroad";
it only needs to be ABOUT someone/something identified as such. None of the following remove a
document from scope once that identification is present: a crime, arrest, or legal case; political
surveillance or suspected revolutionary activity; an administrative/regulatory notice; a financial
request or wire-transfer petition; delegation/travel logistics; visa rules; a social event; a military
organization; examination postponement; a host-country policy or event. Before returning
NO_RELEVANT_SEGMENT, re-read the title and opening lines once more specifically checking for any of
these identifying terms -- this check has been the single largest source of missed relevant documents.

Relevant scope includes:
- Chinese students educated in the United States.
- Chinese students studying abroad generally.
- Returned students or returned-student organizations.
- American-educated, European-educated, Japanese-educated, or otherwise foreign-educated Chinese.
- Foreign-trained PERSONNEL more broadly, not only people explicitly called "students" -- e.g.
  technicians, engineers, or professionals who underwent practical/technical training (實習/受訓/考察)
  abroad, even without the word 學生 appearing.
- Chinese professionals whose authority, politics, career, or conflict is linked to foreign education.
- Study-abroad funding, scholarships, loans, Boxer Indemnity remissions, government missions, or institutions sending students abroad.
- Public debates about Western education, foreign degrees, returned students, or overseas-educated elites in China.
- Economic, legal, or social issues affecting Chinese students or overseas Chinese specifically because of
  their student/overseas status -- e.g. economic exploitation, racial or ethnic discrimination, legal
  disputes, or mistreatment experienced by Chinese students/overseas Chinese abroad.
- A host country's policy or actions toward Chinese students studying there (e.g. Japanese government
  policy or public sentiment toward Chinese students in Japan) -- not just China-side policy toward its
  outgoing students.
- General discussion of the study-abroad phenomenon itself -- trends, numbers, social impact of returned
  students, or sentiment (positive or negative, including anti-foreign or anti-returned-student sentiment)
  connected to study abroad -- even without naming specific individuals.

IMPORTANT -- do not under-apply this scope:
- Once someone is identified as a returned/foreign-educated student, ANY article about them stays in
  scope, even if that specific article covers something else entirely (a legal case, a diplomatic
  incident, a military posting, an administrative dispute). Their foreign-education background is
  what makes it relevant, not whether the article re-explains it.
- Administrative, regulatory, or procedural material -- oversight of students abroad, funding/stipend
  rules, appointment or handover of a student supervisor/inspector, selection procedures for training
  programs -- is relevant even when phrased in dry bureaucratic language rather than as a narrative.
- A title alone stating someone is a returned/foreign-trained student, or naming a study-abroad
  program/subsidy/supervisory office, is sufficient grounds for relevance by itself. Do not reject a
  document just because the body doesn't separately re-state or elaborate on that same fact.

IMPORTANT -- do not over-apply this scope either:
- A delegation, trade/industrial mission, official visit, or group of travelers RETURNING FROM ABROAD
  is NOT automatically relevant just because they came back from the US or elsewhere. Relevance
  requires the trip/people to be about STUDYING -- students, returned students, or education-linked
  funding/policy. A trade delegation, business tour, or diplomatic visit is out of scope even if it
  is textually similar to a "returned student" story (people arriving back in China from America).
- "庚款" (Boxer Indemnity funds) is relevant whenever it is connected to students in ANY way --
  funding, selection, examinations, scholarships, or their suspension/postponement due to war or
  other events. Only exclude an article that is PURELY about bond repayment/finance mechanics with
  NO mention of students, examinations, or scholarships anywhere in it.

FEW-SHOT CALIBRATION EXAMPLES (real cases the model previously got wrong -- study these carefully):

RELEVANCE (surface topic vs. identified status -- the dominant error type):
1. Smoking-related arrest of someone who happens to be a returned/foreign-educated student -> RELEVANT.
2. Personal dispute between a Chinese student abroad and his wife -> RELEVANT.
3. Returned student detained/accused of fraud, or under political surveillance as a suspected
   revolutionary -> RELEVANT -- legal/political trouble involving a returned student is still about one.
4. A dry administrative regulation governing returned students' conduct, or postponing an examination
   for overseas students due to war -> RELEVANT, topic_type: policy.
5. A diplomatic mission's withdrawal affecting Chinese students remaining in that country -> RELEVANT.
6. A social event, meeting, petition for funds, or wire-transfer request from a returned-students'
   association (e.g. 留美同學會, 留美山東同學會, an overseas-Chinese student association 華僑學生會) ->
   RELEVANT, topic_type: organization -- even though the surface topic is "an event" or "a request for
   money", not "study" itself.
7. A delegation of students sent abroad by a political party or government body (e.g. 中央黨部所派留美學生)
   -> RELEVANT -- this is a student delegation, unlike a trade/diplomatic delegation (see the negative
   example above) which is not.
8. US visa regulations covering Chinese travelers generally, where students are among those affected ->
   RELEVANT, topic_type: policy.
9. A social event hosted by a foreign organization for international students that happens to include
   Chinese students among several nationalities -> RELEVANT, topic_type: organization.
10. A Chinese woman who graduated from a US university and went into a film/legal/other career ->
    RELEVANT, topic_type: biography -- her subsequent career doesn't remove her from scope.
11. Foreign-trained technical personnel (e.g. engineers who did practical training in the US) returning
    to China -> RELEVANT even though they are never called "students" (學生) -- see the scope bullet
    on foreign-trained personnel above.
12. A philosophical/moral commentary, or even a humorous aside, that uses "returned student" (留學生)
    to describe someone (even loosely, e.g. joking that self-taught opera fans are like "returned
    students" of gramophone records) -> RELEVANT, topic_type: debate.

SEGMENTATION (extraction precision):
13. A Tokyo-dispatch brevities column with many short items -> MULTIPLE_SEGMENTS, extracting ONLY the
    items themselves about Chinese/returned students in Japan; leave out unrelated items (railway
    construction, legal cases, military disbanding, unrelated funding).
14. OCR duplicates the same relevant announcement twice (transcription artifact) amid otherwise
    unrelated content -> SINGLE_SEGMENT with ONE copy, not MULTIPLE_SEGMENTS.
15. A cohesive roster/list of several returned students' names, hometowns, universities, and fields ->
    SINGLE_SEGMENT (one list = one segment), topic_type: biography.
16. HARD CASE -- a long entertainment/classifieds newspaper page (theater listings, movie reviews,
    opera schedules, ticket prices, shop ads) that is mostly filler, with only one or two short
    passages about a US/foreign-trained individual buried inside (e.g. one sentence on a drama student
    who returned from the US on a Boxer Indemnity scholarship, elsewhere a paragraph on a US-trained
    technical consultant) -> MULTIPLE_SEGMENTS containing ONLY those specific passages, each just long
    enough to cover the relevant person's story -- even though this means discarding the vast majority
    of the document's characters as irrelevant filler. Do not default to WHOLE_DOCUMENT or include
    unrelated movie reviews/theater ads/ticket prices just because they are interspersed with the
    relevant passages -- the relevant text is almost always a small fraction of a page like this.

The Title is given to you separately as metadata below, NOT inside the "Document text" section --
but it is still fully part of the document for the purpose of judging relevance and choosing
segments. Relevant information is very often stated ONLY in the title (a person's degree, country
of study, or student status may appear in the headline and nowhere in the body). Always weigh the
Title as carefully as the body text, and do not require the body to restate what the title already
says -- but do NOT copy the title's wording into segment_text; segment_text should only ever contain
body text.

Rules:
- If the document has NO relevant material in the scope above, return NO_RELEVANT_SEGMENT and segments=[].
- Do not invent relevance. But do not require explicit body-text confirmation of something the title already states plainly.
- If the document is one coherent article and study abroad/Chinese students are only mentioned in passing, keep the ENTIRE document.
- If the document is an OCR/OLR cluster or a brevities/local/personal-news column, extract ONLY the item(s) that are themselves about the relevant scope. Advertisements, corporate/stock/shareholder announcements, wire-service news bulletins on unrelated topics (politics, military, crime, weather, etc.), and other newspaper-page filler are NEVER part of a relevant segment, even when OCR has concatenated them directly onto a relevant article with no visible separator -- exclude them from segment_text entirely, whether that means using SINGLE_SEGMENT/MULTIPLE_SEGMENTS to cut them out or (see below) recognizing the page is not "one coherent article" to begin with.
- If several separated relevant items appear, return multiple segments.
- NEVER include an item in `segments` that is not itself relevant, even if it sits right next to a
  relevant item in the same OCR block (election notices, unrelated local news, ads, wire bulletins).
  Every single entry in `segments` must independently satisfy the relevant scope on its own. Padding
  MULTIPLE_SEGMENTS with a non-relevant neighbor "just in case" is a real mistake that has been
  observed -- if only one item is actually relevant, the correct label is SINGLE_SEGMENT with exactly
  that one item, not MULTIPLE_SEGMENTS with an extra irrelevant one attached.
- Conversely, when a document reports the SAME event from multiple sources or in multiple dispatches
  (e.g. two or three separate wire-service reports on one incident, each from a different
  correspondent/source), find and extract ALL of them if each is independently relevant -- do not
  stop scanning after finding the first one or two. Read to the actual end of the document before
  finalizing MULTIPLE_SEGMENTS; missing a later relevant block has been observed.
- COPY segment_text VERBATIM from the document -- character for character, same punctuation, same
  spacing, same traditional-vs-simplified characters as the source. Do NOT paraphrase, summarize,
  correct, modernize, or otherwise regenerate the text in your own words, even slightly -- this has
  been observed happening and silently corrupts the historical record. If you cannot reproduce a
  passage exactly, choose a shorter, more certain span rather than approximating a longer one.
- Segment boundaries must be continuous spans from the document.
- Each segment must include its OWN full extent: any short framing/heading phrase or dateline that
  introduces that specific item (often a brief clause ending in 。 right before the narrative starts),
  through to that item's natural concluding sentence. Do not start your segment mid-narrative (skipping
  the item's own opening clause) and do not stop before its natural conclusion -- both have been
  observed mistakes. When in doubt, err toward including one extra boundary sentence rather than cutting
  into the item. This applies to WHOLE_DOCUMENT too: if a short news item is itself introduced by scene-
  setting sentences (e.g. an unrelated-seeming preamble about a shipping company's office before the
  actual passenger list), that preamble is part of the SAME continuous item, not a separate one to trim.

CRITICAL -- your segmentation_label MUST agree with your own segmentation_explanation:
- Before answering, check: does my label match what I just wrote in my explanation? If the
  explanation says the document is NOT about the relevant scope, discusses unrelated matters
  (finance, administration, etc.), or otherwise explains why nothing qualifies, the label MUST be
  NO_RELEVANT_SEGMENT -- never WHOLE_DOCUMENT/SINGLE_SEGMENT/MULTIPLE_SEGMENTS in that case. A
  positive label (anything other than NO_RELEVANT_SEGMENT) requires an explanation that actually
  states why the content IS relevant.
- Likewise: if your explanation describes MULTIPLE distinct items, references, or people (e.g. "the
  document contains multiple references to...", "several students are named...", "two separate
  reports..."), your label must be SINGLE_SEGMENT or MULTIPLE_SEGMENTS with that many segments --
  never WHOLE_DOCUMENT. Saying "multiple" in your reasoning and then choosing WHOLE_DOCUMENT is a
  real, observed contradiction; re-read your own explanation before finalizing the label.
- The reverse also applies, and matters just as much: before writing "no identifying terms found" or
  similar in a NO_RELEVANT_SEGMENT explanation, re-scan the title character by character. Titles in
  this corpus are short -- missing a term that is plainly present in the title (e.g. 留學, 出洋, 留美,
  庚款, 同學會) is a real error, not a judgment call, and has been observed happening.

CRITICAL -- WHOLE_DOCUMENT vs SINGLE_SEGMENT are NOT interchangeable defaults:
- SINGLE_SEGMENT / MULTIPLE_SEGMENTS are ONLY for a document that is a container of several
  genuinely separate, unrelated news items (e.g. an OCR'd brevities/local-news column stitching
  together unrelated one-line notices), where only some item(s) are relevant.
- If the document is one coherent article, notice, announcement, or report -- even a short one,
  even if some sentences are incidental (names, dates, logistics) rather than literally about study
  abroad -- use WHOLE_DOCUMENT. Do not carve a "relevant excerpt" out of a single continuous piece
  of writing; that is the most common mistake to avoid.
- BUT: "one coherent article" means the input is genuinely just that one piece of writing.
  Advertisements, corporate/shareholder/stock announcements, and wire-service news bulletins on
  clearly unrelated topics are NEVER part of the same "article" as a relevant piece next to them --
  even when OCR ran them together into one block of text with no visible separator. If the input
  text contains a relevant article followed or surrounded by such unrelated material, that is a
  container of multiple items, NOT one coherent article -- use SINGLE_SEGMENT/MULTIPLE_SEGMENTS to
  extract ONLY the relevant article's own text, excluding the ads/bulletins entirely.
- Test 1 (for using SINGLE_SEGMENT/MULTIPLE_SEGMENTS instead of WHOLE_DOCUMENT): if producing your
  segment would mean cutting a sentence out of the middle of one continuous narrative/report, use
  WHOLE_DOCUMENT instead. But if what you'd be excluding is a distinct item (an ad, a bulletin, an
  unrelated notice) rather than a mid-sentence cut, exclude it -- that is correct, not a mistake.
- Test 2: before choosing WHOLE_DOCUMENT, scan the full input text once more specifically for
  advertisement/announcement/bulletin markers (prices, "each/per", company names with 公司/商店/藥房/
  銀行, event/meeting announcements unrelated to the topic, wire datelines like a city name in
  parentheses followed by unrelated news). If you find any, they must be excluded -- do not choose
  WHOLE_DOCUMENT if it would sweep them in.

- When you keep the ENTIRE document (WHOLE_DOCUMENT), DO NOT reproduce the document text. Set segment_text to an empty string ""; the full text is inserted automatically from the input. Still provide topic_type and relevance_to_original_topic.

For each relevant segment, also provide:
- topic_type: an ARRAY of one or more of the following fixed labels (lowercase, verbatim, no new
  categories) that genuinely apply. Most segments need only one, but use more when genuinely
  warranted -- e.g. an organization's event where a named individual's biography is also reported ->
  ["organization","biography"]; a policy notice that itself sparked public debate ->
  ["policy","debate"]. Do not pad with extra labels that don't really fit. "other" must never be
  combined with a substantive label -- if any of policy/debate/biography/organization applies, do not
  also include "other".
{TOPIC_LABEL_DESCRIPTIONS}
- relevance_to_original_topic: RELEVANT_US if the segment mentions the United States/America ANYWHERE
  in the relevant content -- even if it also discusses other countries, or is a comparison/list
  including the US (e.g. "returned students from Europe and America" -> RELEVANT_US; a scholarship
  roster naming a US destination among several -> RELEVANT_US). Use RELEVANT_NON_US whenever the US
  is NOT explicitly named among the relevant countries -- e.g. it concerns the UK, Japan, or Europe
  specifically, or uses a general phrase like "the West and East" (東西洋) / "abroad" without naming
  the US, or names no country at all. Do not default to RELEVANT_US when uncertain -- only choose it
  when the US is textually present.

OUTPUT BREVITY: give exactly ONE segmentation_confidence (0.0-1.0) and keep segmentation_explanation to ONE short sentence. Always fill in segmentation_explanation, even for NO_RELEVANT_SEGMENT -- state briefly why nothing qualified.

Return only valid JSON."""

# --- Trimmed variant: a deliberately condensed "highest-signal core" of the
# prompt above, built to A/B test whether a shorter prompt is actually more
# reliable than the long, heavily-annotated one (which has grown across many
# rounds of fixes and may be diluting the model's attention rather than
# helping -- see PATCH_VERSION history). Differences from the full version:
#   - The ABSOLUTE RULE + "do not under-apply" + several few-shot examples in
#     the full prompt all restate the same "surface topic vs identified
#     status" point 3+ times; here it's stated once, plus a shorter, curated
#     set of examples (6, down from 16) covering distinct failure categories
#     rather than many near-duplicate scenarios.
#   - relevance_to_original_topic's verbose US-vs-non-US definition is cut
#     to one line: the actual label is fully overridden deterministically in
#     repair()/relevance_with_us_backstop() regardless of what the model
#     outputs here, so a long explanation of it is wasted prompt budget.
#   - gender_mentioned is not mentioned at all (also fully deterministic,
#     computed in repair(), never requested from the model in either variant).
SYSTEM_PROMPT_TRIMMED = f"""You are a historian's segmentation assistant for OCR/OLR historical newspaper data.

Your task is NOT to summarize. Delineate the largest semantic unit(s) in a document that contain
relevant study-abroad context, then classify each relevant unit.

ABSOLUTE RULE: if ANY part of the document (title or body) identifies a person, group, or
organization as a returned/overseas/foreign-trained student, a study-abroad alumni/students'
association, or other foreign-trained personnel (terms like 留學生, 留美/英/日/德/法/俄/歐, 游學, 出洋,
同學會, 華僑學生, 庚款生, or a description of practical/technical training abroad), the document IS
relevant -- no matter what the rest of the article discusses. It only needs to be ABOUT someone/
something so identified; it does not need to be mainly ABOUT studying. None of the following remove a
document from scope once that identification is present: a crime/legal case, political surveillance,
an administrative notice, a financial request, delegation/travel logistics, visa rules, a social
event, examination postponement, a host-country policy. Before returning NO_RELEVANT_SEGMENT, re-read
the title once more for these terms -- missing one that is plainly present has been the largest
source of errors.

Relevant scope: Chinese students/returned students educated anywhere abroad; foreign-trained
personnel more broadly (not just people called "students" -- engineers/technicians with practical
training abroad count too); Chinese professionals whose career/politics/conflict is linked to foreign
education; study-abroad funding/scholarships/Boxer Indemnity when connected to students in any way;
public debate about study abroad or returned students; economic/legal/social issues (discrimination,
exploitation, legal trouble) affecting Chinese students/overseas Chinese because of that status; a
host country's policy toward Chinese students there; general discussion of the study-abroad
phenomenon even without named individuals.

NOT in scope: a delegation/trade mission/diplomatic visit returning from abroad that is not itself
about studying (a trade tour is not relevant just because it's textually similar to a "returned
student" story). An article purely about Boxer Indemnity bond repayment/finance mechanics with zero
connection to students/examinations/scholarships anywhere in it.

Segmentation:
- One coherent article/notice/report, even a short one -> WHOLE_DOCUMENT. Do not carve an excerpt out
  of one continuous piece of writing, even if a scene-setting preamble precedes the core item.
- A brevities/local-news column or OCR cluster of several genuinely separate, unrelated items -> use
  SINGLE_SEGMENT/MULTIPLE_SEGMENTS and extract ONLY the relevant item(s).
- Every entry in `segments` must independently be relevant on its own -- never pad MULTIPLE_SEGMENTS
  with a non-relevant neighbor. If only one item is relevant, the label is SINGLE_SEGMENT with exactly
  that one item, nothing else attached.
- If the SAME event is reported in multiple separate dispatches, find and extract ALL relevant ones --
  read to the actual end of the document, don't stop after the first one or two.
- Advertisements, corporate/stock announcements, and wire bulletins on unrelated topics are NEVER part
  of a relevant segment, even when OCR concatenated them onto a relevant article with no separator --
  exclude them, and recognize that such a page is a container of multiple items, not "one article".
- COPY segment_text VERBATIM -- character for character, same punctuation/spacing/script as the
  source. Never paraphrase, summarize, or regenerate it in your own words, even slightly.
- Each segment must include its own full extent -- its own opening framing/dateline clause through to
  its natural concluding sentence -- not just the narrative middle. When in doubt, include one extra
  boundary sentence rather than cut into the item.
- Your segmentation_label must agree with your segmentation_explanation: an explanation saying the
  document is unrelated requires NO_RELEVANT_SEGMENT; a positive label requires an explanation that
  actually states why the content is relevant; an explanation describing MULTIPLE items/references
  requires SINGLE_SEGMENT/MULTIPLE_SEGMENTS, never WHOLE_DOCUMENT.
- If OCR duplicates the same item twice, that is still one segment, not two.

FEW-SHOT EXAMPLES (distinct failure categories, not exhaustive):
1. A crime/legal/political-surveillance story about a returned student -> RELEVANT (their status is
   what matters, not the story's surface topic).
2. A social event, meeting, or funding petition from a returned-students' association (e.g. 留美同學會)
   -> RELEVANT, topic_type: organization -- even though the surface topic is "an event", not "study".
3. Foreign-trained technical personnel (never called "學生") returning to China -> RELEVANT.
4. A long entertainment/classifieds page (theater/movie listings, ticket prices) that is mostly
   filler, with only one or two short passages about a foreign-trained individual -> MULTIPLE_SEGMENTS
   containing ONLY those specific passages, discarding the rest even though it's the majority of the
   page. Do not default to WHOLE_DOCUMENT here.
5. A Tokyo-dispatch brevities column with many short items -> MULTIPLE_SEGMENTS, extracting only the
   items about Chinese/returned students in Japan, leaving out unrelated items (railway, legal, military).
6. A cohesive roster/list of several returned students' names and fields -> SINGLE_SEGMENT (one list =
   one segment), topic_type: biography.

The Title is given separately as metadata below, not inside "Document text" -- but weigh it as
carefully as the body; do not require the body to restate what the title states, and do not copy the
title's wording into segment_text.

For each relevant segment, provide:
- topic_type: an ARRAY of one or more labels that genuinely apply (most segments need only one; e.g.
  an organization event that also reports a named biography -> ["organization","biography"]).
  "other" is never combined with a substantive label. Labels (lowercase, verbatim):
{TOPIC_LABEL_DESCRIPTIONS}
- relevance_to_original_topic: RELEVANT_US if the US is mentioned anywhere in the relevant content
  (alone or alongside other countries); otherwise RELEVANT_NON_US. Do not default to RELEVANT_US when
  uncertain.

When you keep the ENTIRE document (WHOLE_DOCUMENT), set segment_text to "" -- the full text is
inserted automatically.

OUTPUT BREVITY: exactly one segmentation_confidence (0.0-1.0) and one short segmentation_explanation
sentence, always filled in even for NO_RELEVANT_SEGMENT.

Return only valid JSON."""

# Which variant is actually used for inference -- selectable per run via the
# PROMPT_VARIANT env var so both can be tested on the same input sample and
# compared. See the comparison cell in the notebook.
PROMPT_VARIANT = os.environ.get("PROMPT_VARIANT", "full").strip().lower()
if PROMPT_VARIANT not in ("full", "trimmed"):
    raise ValueError(f"PROMPT_VARIANT must be 'full' or 'trimmed', got {PROMPT_VARIANT!r}")
SYSTEM_PROMPT = SYSTEM_PROMPT_TRIMMED if PROMPT_VARIANT == "trimmed" else SYSTEM_PROMPT_FULL

JSON_REQUIREMENTS = """
Return ONLY one valid JSON object. Use exactly these keys:

{
  "doc_id": "string",
  "segmentation_label": "WHOLE_DOCUMENT or SINGLE_SEGMENT or MULTIPLE_SEGMENTS or NO_RELEVANT_SEGMENT",
  "segmentation_confidence": 0.0,
  "segmentation_explanation": "ONE short sentence explaining the segmentation decision",
  "segments": [
    {
      "segment_index": 1,
      "segment_text": "continuous exact span from the document for a PARTIAL segment; empty string when WHOLE_DOCUMENT",
      "topic_type": ["policy", "debate", "biography", "organization", "other" -- ARRAY of 1+ that genuinely apply],
      "relevance_to_original_topic": "RELEVANT_US or RELEVANT_NON_US"
    }
  ]
}

When you keep the whole document, leave segment_text empty:
{
  "doc_id": "string",
  "segmentation_label": "WHOLE_DOCUMENT",
  "segmentation_confidence": 0.0,
  "segmentation_explanation": "one sentence",
  "segments": [
    {"segment_index": 1, "segment_text": "", "topic_type": ["policy"], "relevance_to_original_topic": "RELEVANT_US"}
  ]
}

For a no-relevant-content document:
{
  "doc_id": "string",
  "segmentation_label": "NO_RELEVANT_SEGMENT",
  "segmentation_confidence": 0.0,
  "segmentation_explanation": "one sentence explaining why nothing qualified",
  "segments": []
}
"""


PATCH_VERSION = "batch_resume_v15_verbatim_recheck_relevance_evidence"


def now():
    return time.strftime("%Y-%m-%d %H:%M:%S")

def clean_text(x: Any) -> str:
    if pd.isna(x):
        return ""
    return re.sub(r"\s+", " ", str(x)).strip()

def one_sentence(x: Any) -> str:
    """Return a one-sentence explanation.

    The original version returned "" whenever the text had no ASCII
    terminal punctuation (.!?), which silently discarded perfectly good
    Chinese-punctuated or unpunctuated explanations and made real model
    failures indistinguishable from "the model just phrased it differently".
    We now also recognise CJK terminal punctuation, and fall back to the
    trimmed text itself (instead of an empty string) when no terminator is
    found at all.
    """
    s = re.sub(r"\s+", " ", str(x or "")).strip()
    if not s:
        return ""
    m = re.search(r"(.+?[.!?。！？])(\s|$)", s)
    if m:
        return m.group(1).strip()
    return s

def normalize_topic_type(x: Any) -> str:
    s = str(x or "").strip().lower()
    if s in TOPIC_LABELS:
        return s
    # Fuzzy fallback for wording/pluralization variants the model might use
    # instead of the exact label, e.g. "Policies", "biographical",
    # "organizations", "debates". A shared 5-character prefix reliably
    # catches these without false-matching unrelated free text.
    PREFIX_LEN = 5
    s_prefix = s[:PREFIX_LEN]
    for label in TOPIC_LABELS:
        if label in s or s in label or label[:PREFIX_LEN] == s_prefix:
            return label
    return "other"

_TOPIC_LABEL_ORDER = ["policy", "debate", "biography", "organization", "other"]

def normalize_topic_types(x: Any) -> str:
    """topic_type can now be MULTIPLE labels (e.g. an organization's event
    that also reports a named individual's biography -> policy+debate,
    biography+organization, etc.). Accepts a JSON array (the requested
    schema) or, as a fallback for a model that doesn't strictly follow it, a
    comma/semicolon/slash-separated string. Returns a '; '-joined string in
    canonical label order, deduplicated. 'other' is dropped whenever at
    least one substantive label is also present -- it should never appear
    combined with a real label, only ever alone as a genuine fallback."""
    if isinstance(x, list):
        raw_items = x
    elif isinstance(x, str):
        raw_items = re.split(r"[;,/、]", x)
    else:
        raw_items = []

    found = {normalize_topic_type(item) for item in raw_items if str(item or "").strip()}
    if not found:
        found = {"other"}
    if len(found) > 1:
        found.discard("other")

    return "; ".join(l for l in _TOPIC_LABEL_ORDER if l in found)

def topic_type_flags(joined_topic_type: str) -> Dict[str, bool]:
    """One-hot boolean columns derived from the joined topic_type string,
    e.g. "organization; biography" -> only topic_organization and
    topic_biography True. Makes multi-label filtering/aggregation in the CSV
    (df[df.topic_biography]) trivial without parsing the joined string."""
    labels = {l.strip() for l in joined_topic_type.split(";") if l.strip()}
    return {f"topic_{l}": (l in labels) for l in TOPIC_LABELS}

# gender_mentioned is deliberately NOT asked of the model (removed from the
# JSON schema): the model's own judgement here was unreliable in both
# directions -- false positives on gender-neutral/male terms like 兄弟
# ("brothers", e.g. as part of a company name) and 先生 (an honorific, not
# necessarily gendered), and false negatives even on unambiguous markers
# like 女士 attached to a named woman. The user only wants the female side
# of gender specifically, which is a well-scoped lexical-matching problem --
# so it's computed deterministically here instead, which is both more
# reliable and saves output tokens on every single call.
_FEMALE_MARKERS = [
    "女士", "女生", "女子", "女性", "女兒", "女孩", "婦女", "小姐", "姑娘", "閨秀", "淑女",
    "夫人", "太太", "新娘", "未婚妻", "妻子", "母親", "令堂", "令愛", "令媛",
    "姊妹", "姐妹", "胞妹", "孫女", "侄女", "外甥女", "女校", "女師", "女中", "巾幗", "閨女",
]
# NOTE: bare single characters (妹/姊/姐/妻/母) were deliberately removed --
# they matched inside unrelated idioms and boilerplate (e.g. 母寧/毋寧
# "rather", 母校 "alma mater", and the stock public-notice salutation
# 父老姑姊妹 "elders and sisters") and were a real source of false positives
# seen in practice. The remaining markers are all >=2-character compounds
# specific enough to reliably indicate an actual woman/girl is being
# referred to.

def find_female_markers(*texts: Any) -> list[str]:
    """Returns every _FEMALE_MARKERS term found in the text, in the order
    they're listed (dedup'd), so callers can report both whether a woman is
    mentioned AND which term(s) justify that."""
    combined = " ".join(str(t or "") for t in texts)
    return [m for m in _FEMALE_MARKERS if m in combined]

def mentions_women(*texts: Any) -> bool:
    """True only if the text explicitly refers to a woman/girl -- see
    _FEMALE_MARKERS. Deliberately narrow and female-specific per the
    research focus; does not flag male or gender-neutral terms."""
    return bool(find_female_markers(*texts))

# High-precision markers for "this document almost certainly involves a
# returned/foreign-trained student or a student/alumni organization" -- used
# ONLY to catch NO_RELEVANT_SEGMENT judgements worth a second look (see
# call_model_with_relevance_recheck), not to auto-override the model. These
# are deliberately restricted to terms that are unambiguous in this corpus;
# broader/vaguer terms (e.g. bare 出洋 "going abroad", which also covers
# trade/diplomatic travel that is explicitly OUT of scope) are intentionally
# excluded to avoid forcing a recheck on genuinely irrelevant documents.
_RELEVANCE_RECHECK_MARKERS = [
    "留學", "出洋留學", "留美", "留英", "留日", "留德", "留法", "留俄", "留歐",
    "游學", "遊學", "庚款", "同學會", "華僑學生", "官費生", "自費生",
]

def find_relevance_recheck_marker(*texts: Any) -> str:
    """Returns the first high-precision relevance marker found, or "" if
    none. Used to flag a NO_RELEVANT_SEGMENT judgement for a second,
    targeted look rather than trusting it blindly -- see
    call_model_with_relevance_recheck()."""
    combined = " ".join(str(t or "") for t in texts)
    for m in _RELEVANCE_RECHECK_MARKERS:
        if m in combined:
            return m
    return ""

def extract_json(text: str) -> Dict[str, Any]:
    text = text.strip()
    # Strip any leftover reasoning before the JSON.
    if "</think>" in text:
        text = text.split("</think>")[-1]
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.I)
    text = re.sub(r"\s*```$", "", text)

    try:
        return json.loads(text)
    except Exception:
        pass

    start = text.find("{")
    if start < 0:
        raise ValueError("No JSON object found")

    depth = 0
    in_str = False
    esc = False

    for i, ch in enumerate(text[start:], start=start):
        if esc:
            esc = False
            continue
        if ch == "\\" and in_str:
            esc = True
            continue
        if ch == '"':
            in_str = not in_str
            continue
        if in_str:
            continue
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return json.loads(text[start:i + 1])

    raise ValueError("Could not parse JSON")



_US_DIRECT_MARKERS = ["美國", "美洲", "留美", "游美", "遊美", "赴美", "駐美", "旅美", "僑美", "在美",
                      "United States", "U.S.", "U. S.", "USA", "America"]
# Single-character country abbreviations commonly paired with 美 in this
# corpus to signal "the US alongside/compared with other countries" (e.g.
# 歐美 "Europe-America", 英美 "Britain-America" as in "英美學金"/scholarships
# to Britain-and-America). A fixed marker list alone missed these compounds.
_US_PAIR_CHARS = "英法德俄日中韓意荷比瑞葡奧西歐"
_US_PAIR_RE = re.compile(f"[{_US_PAIR_CHARS}]美(?!洲|國)|美[{_US_PAIR_CHARS}]")

def mentions_us(*texts: Any) -> bool:
    """Deterministic backstop for relevance_to_original_topic: if the US is
    named anywhere in the relevant text/title -- alone, or paired/compared
    with other countries (歐美, 英美, etc.) -- the segment must be
    RELEVANT_US. Don't just trust the model's classification when this is
    checkable directly."""
    combined = " ".join(str(t or "") for t in texts)
    if any(m in combined for m in _US_DIRECT_MARKERS):
        return True
    return bool(_US_PAIR_RE.search(combined))

def find_us_markers(*texts: Any) -> list[str]:
    """Returns every US-indicating marker/pattern match found in the text,
    mirroring find_female_markers -- so callers can report exactly which
    term(s) justify a RELEVANT_US classification, the same way
    gender_evidence reports which term(s) justify gender_mentioned."""
    combined = " ".join(str(t or "") for t in texts)
    found = [m for m in _US_DIRECT_MARKERS if m in combined]
    for m in _US_PAIR_RE.finditer(combined):
        token = m.group(0)
        if token not in found:
            found.append(token)
    return found

def relevance_with_us_backstop(raw_relevance: Any, seg_title: str, seg_text: str) -> str:
    """Fully deterministic, symmetric: the label always follows what's
    textually checkable, rather than trusting the model's classification
    in either direction.
      - US textually present (alone, or paired with other countries like
        歐美/英美) -> always RELEVANT_US, regardless of what the model said.
      - US textually absent -> always RELEVANT_NON_US, even if the model
        said RELEVANT_US (this was observed happening on UK-only articles,
        and on generic "留學" mentions with no country specified at all --
        the model should not default to US when it isn't evidenced)."""
    has_us = mentions_us(seg_title, seg_text)
    return "RELEVANT_US" if has_us else "RELEVANT_NON_US"

def relevance_evidence_for(relevance_label: str, doc_title: str, seg_text: str) -> str:
    """Mirrors gender_evidence: when the final label is RELEVANT_US, report
    the specific matched term(s) that justify it. Empty for RELEVANT_NON_US,
    since that label reflects an absence of evidence rather than positive
    evidence of its own."""
    if relevance_label != "RELEVANT_US":
        return ""
    return "; ".join(find_us_markers(doc_title, seg_text))

def is_verbatim(segment_text: str, original: str) -> bool:
    """Deterministic check: is segment_text actually a contiguous substring
    of the source document (given the whitespace normalization clean_text()
    already applies to both), or did the model paraphrase/regenerate it
    instead of copying it? This does NOT attempt to fix a non-verbatim
    segment -- fuzzy-correcting risks introducing new errors of its own --
    it only flags it (segment_verbatim=False) so it can be found and
    reviewed rather than silently trusted."""
    if not segment_text:
        return True
    return segment_text in original

def dedupe_segments(segments: list) -> list:
    """Collapse near-duplicate segments caused by OCR repeating the same
    passage twice (a transcription artifact, observed in practice) -- e.g.
    the model returning the identical announcement as two "segments" when
    it only occurs once conceptually. Deliberately conservative: only merges
    when one segment's text is equal to, or entirely contained within,
    another's (and long enough that this can't be a coincidental short-string
    match), so genuinely distinct short items are never merged."""
    kept: list = []
    seen_texts: list = []
    for seg in segments:
        text = str(seg.get("segment_text", "") or "").strip()
        is_dup = bool(text) and any(
            text == prev or (len(text) > 20 and (text in prev or prev in text))
            for prev in seen_texts
        )
        if not is_dup:
            kept.append(seg)
            seen_texts.append(text)
    return kept

def repair(obj: Dict[str, Any], row: Dict[str, Any]) -> Dict[str, Any]:
    doc_id = str(row.get("doc_id", ""))
    obj["doc_id"] = str(obj.get("doc_id") or doc_id)

    # A PROCESSING_ERROR is an infra/parsing failure, not a content judgement
    # -- pass it through as-is rather than running it through the normal
    # WHOLE_DOCUMENT/SINGLE_SEGMENT/etc. relabeling logic below, which would
    # otherwise silently launder it into what looks like a real decision.
    if obj.get("segmentation_label") == "PROCESSING_ERROR":
        obj.setdefault("segments", [])
        obj.setdefault("segmentation_confidence", 0.0)
        obj.setdefault("segmentation_explanation", obj.get("_parse_error") or "")
        return obj

    # `row["text"]` is pure body text -- title is intentionally NOT merged in
    # (see main()), so this is exactly what should land in segment_text.
    original = clean_text(row.get("text", ""))
    doc_title = str(row.get("title", "") or "")

    # ---- Segmentation (one confidence, one sentence) ----
    if not isinstance(obj.get("segments"), list):
        obj["segments"] = []

    try:
        sc = max(0.0, min(1.0, float(obj.get("segmentation_confidence", 0.0))))
    except Exception:
        sc = 0.0

    # NOTE: keep_entire_document was removed -- it was redundant with
    # segmentation_label == "WHOLE_DOCUMENT" and cost extra output tokens on
    # every single call for no benefit. WHOLE_DOCUMENT-ness is now derived
    # solely from segmentation_label (as reported by the model, or as set by
    # the coverage-based promotion below).
    old_label = obj.get("segmentation_label")
    want_whole = old_label == "WHOLE_DOCUMENT"

    if want_whole:
        src = obj["segments"][0] if obj["segments"] and isinstance(obj["segments"][0], dict) else {}
        seg_text = original  # grabbed from the input, never regenerated
        female_terms = find_female_markers(doc_title, seg_text)
        topic_type_joined = normalize_topic_types(src.get("topic_type"))
        final_relevance = relevance_with_us_backstop(src.get("relevance_to_original_topic"), doc_title, seg_text)
        obj["segments"] = [{
            "segment_index": 1,
            "segment_text": seg_text,
            "segment_verbatim": True,  # always true: text is taken from the input, never model-generated
            "topic_type": topic_type_joined,
            **topic_type_flags(topic_type_joined),
            "relevance_to_original_topic": final_relevance,
            "relevance_evidence": relevance_evidence_for(final_relevance, doc_title, seg_text),
            "gender_mentioned": bool(female_terms),
            "gender_evidence": "; ".join(female_terms),
        }]
        obj["segmentation_label"] = "WHOLE_DOCUMENT"
    else:
        for i, seg in enumerate(obj["segments"], start=1):
            if not isinstance(seg, dict):
                seg = {}
                obj["segments"][i - 1] = seg
            for dead in ("boundary_confidence", "boundary_rationale", "excluded_content_summary", "segment_title"):
                seg.pop(dead, None)
            seg["segment_index"] = int(seg.get("segment_index") or i)
            seg_text = clean_text(seg.get("segment_text", ""))
            seg["segment_text"] = seg_text
            seg["segment_verbatim"] = is_verbatim(seg_text, original)
            seg["topic_type"] = normalize_topic_types(seg.get("topic_type"))
            seg.update(topic_type_flags(seg["topic_type"]))
            seg["relevance_to_original_topic"] = relevance_with_us_backstop(seg.get("relevance_to_original_topic"), doc_title, seg_text)
            seg["relevance_evidence"] = relevance_evidence_for(seg["relevance_to_original_topic"], doc_title, seg_text)
            female_terms = find_female_markers(doc_title, seg_text)
            seg["gender_mentioned"] = bool(female_terms)
            seg["gender_evidence"] = "; ".join(female_terms)

        # Collapse OCR-duplicated segments (the same passage appearing twice)
        # into one before counting segments for the label -- otherwise a
        # single cohesive item gets mislabeled MULTIPLE_SEGMENTS just because
        # it was transcribed twice. Re-index afterward so segment_index stays
        # contiguous.
        obj["segments"] = dedupe_segments(obj["segments"])
        for i, seg in enumerate(obj["segments"], start=1):
            seg["segment_index"] = i

        # Safety net for the most common mislabeling we've observed: the
        # model calling something SINGLE_SEGMENT when the "segment" it
        # extracted is actually nearly the whole document -- i.e. a single
        # coherent article that should have been WHOLE_DOCUMENT. Don't just
        # trust the label; if the extracted span covers almost all of the
        # source text, treat it as what it actually is.
        # IMPORTANT: coverage is a meaningless/misleading signal for a
        # non-verbatim (paraphrased/regenerated) segment -- a regenerated
        # span in a different phrasing (or a different script/language
        # entirely) can easily be LONGER than the source by raw character
        # count despite matching nothing, which would otherwise trigger a
        # false promotion here. Only promote when the segment text is also
        # confirmed verbatim.
        WHOLE_DOC_COVERAGE_THRESHOLD = 0.85
        if len(obj["segments"]) == 1 and original:
            seg0 = obj["segments"][0]
            coverage = len(seg0.get("segment_text", "")) / max(len(original), 1)
            if coverage >= WHOLE_DOC_COVERAGE_THRESHOLD and seg0.get("segment_verbatim", False):
                # segment_text is expanding from seg0's (possibly partial)
                # text to the full `original` -- recompute both the US
                # backstop and gender markers against the actual final text
                # rather than copying seg0's now-stale values.
                promoted_female_terms = find_female_markers(doc_title, original)
                promoted_relevance = relevance_with_us_backstop(seg0["relevance_to_original_topic"], doc_title, original)
                obj["segments"] = [{
                    "segment_index": 1,
                    "segment_text": original,
                    "segment_verbatim": True,  # promoted to WHOLE_DOCUMENT: text is now taken from the input, never model-generated
                    "topic_type": seg0["topic_type"],
                    **topic_type_flags(seg0["topic_type"]),
                    "relevance_to_original_topic": promoted_relevance,
                    "relevance_evidence": relevance_evidence_for(promoted_relevance, doc_title, original),
                    "gender_mentioned": bool(promoted_female_terms),
                    "gender_evidence": "; ".join(promoted_female_terms),
                }]
                obj["segmentation_label"] = "WHOLE_DOCUMENT"
                obj["_promoted_to_whole_document"] = True

        if obj.get("_promoted_to_whole_document"):
            pass  # label already correctly set above
        elif not obj["segments"]:
            obj["segmentation_label"] = "NO_RELEVANT_SEGMENT"
        elif len(obj["segments"]) == 1:
            obj["segmentation_label"] = "SINGLE_SEGMENT"
        else:
            obj["segmentation_label"] = "MULTIPLE_SEGMENTS"

    obj["segmentation_confidence"] = sc
    obj["segmentation_explanation"] = one_sentence(obj.get("segmentation_explanation"))

    return obj

OLLAMA_HOST = os.environ.get("OLLAMA_HOST", "http://127.0.0.1:11434").rstrip("/")
OLLAMA_MODEL = os.environ.get("OLLAMA_MODEL", "qwen2.5:7b")
# Ollama defaults num_ctx to 2048 tokens regardless of what the model can
# actually handle -- far too small once the system prompt + JSON schema +
# a real document are all in the context. This was a likely contributor to
# the earlier "confidence=0, no explanation" failures: the prompt or the
# expected output silently got truncated by Ollama, not by our own MAX_CHARS.
OLLAMA_NUM_CTX = int(os.environ.get("OLLAMA_NUM_CTX", "32768"))

def wait_for_ollama(timeout: int = 60) -> bool:
    import requests
    start = time.time()
    last_err = None
    while time.time() - start < timeout:
        try:
            r = requests.get(f"{OLLAMA_HOST}/api/tags", timeout=5)
            if r.ok:
                names = [m.get("name") for m in r.json().get("models", [])]
                print(f"[{now()}] OLLAMA_REACHABLE host={OLLAMA_HOST} models={names}", flush=True)
                if OLLAMA_MODEL not in names:
                    print(f"[{now()}] WARNING: {OLLAMA_MODEL} not in `ollama list` output -- did the pull finish, or is the tag different?", flush=True)
                return True
        except Exception as e:
            last_err = e
        time.sleep(2)
    print(f"[{now()}] OLLAMA_UNREACHABLE host={OLLAMA_HOST} last_err={last_err}", flush=True)
    return False

def smart_truncate(text: str, max_chars: int) -> str:
    """Head+tail truncation that tries to cut on a sentence boundary near the
    midpoint instead of slicing mid-sentence. This does not fix the
    confidence=0 issue by itself (that also happens on short documents, see
    PATCH_VERSION notes / README), but it reduces the odds that a genuinely
    relevant clause gets split in half and made unreadable.
    """
    if len(text) <= max_chars:
        return text
    half = max_chars // 2
    head, tail = text[:half], text[-half:]
    m = re.search(r"[。！？.!?]", head[::-1])
    if m:
        cut = half - m.start()
        head = head[:cut]
    return head + "\n\n[... DOCUMENT TRUNCATED ...]\n\n" + tail

def format_segments_for_verification(segments: list) -> str:
    if not segments:
        return "(no segments extracted)"
    lines = []
    for seg in segments:
        idx = seg.get("segment_index", "?")
        txt = str(seg.get("segment_text", "") or "")
        lines.append(f"[segment_index {idx}]\n{txt}")
    return "\n\n".join(lines)

def call_model(row: Dict[str, Any], max_new_tokens_override: int | None = None, relevance_hint: str | None = None, verify_segments: list | None = None, verbatim_hint: list | None = None) -> Dict[str, Any]:
    import requests

    text = row["text"]
    max_chars = int(os.environ.get("MAX_CHARS", "20000"))
    text = smart_truncate(text, max_chars)

    hint_block = ""
    if relevance_hint:
        hint_block = f"""

IMPORTANT RECONSIDERATION: on a first pass, this document was judged NO_RELEVANT_SEGMENT, but its
title or text contains the term '{relevance_hint}', which this corpus's scope guide identifies as a
strong returned/study-abroad marker (see the ABSOLUTE RULE near the start of your instructions).
Before finalizing NO_RELEVANT_SEGMENT again, explicitly check whether '{relevance_hint}' identifies a
person, group, or organization as a returned/foreign-trained student or alumni association somewhere
in the title or text. If so, correct your answer to reflect relevance. If, after this specific check,
you still find no such identification (e.g. the term appears in an unrelated phrase), you may confirm
NO_RELEVANT_SEGMENT -- but say explicitly in your explanation that you checked for '{relevance_hint}'
and found it does not identify a relevant person/group in this case."""

    if verify_segments is not None:
        first_pass_block = format_segments_for_verification(verify_segments)
        hint_block += f"""

VERIFICATION PASS: you already produced a first-pass segmentation for this document, shown below.
Before finalizing, re-check specifically for these three known mistakes:
1. BOUNDARIES: does each segment start with its own natural opening (not mid-sentence, not skipping a
   short framing/dateline clause) and end at its own natural conclusion (not cut short)? Correct any
   that don't -- expand the span to cover the item's true start/end.
2. COMPLETENESS: does the rest of the document (outside these segments) contain any OTHER relevant
   item that was missed? If the same event is reported in multiple dispatches, check that ALL of them
   were captured, not just the first one or two. Add any missed item as an additional segment.
3. PURITY: is any of these segments NOT actually relevant on its own (e.g. an unrelated item that got
   attached alongside a real one)? If so, drop it -- never keep an irrelevant segment just because a
   first pass included it.

First-pass extraction:
{first_pass_block}

If the first-pass extraction was already fully correct, return it unchanged. Return your final,
verified answer using the exact same JSON schema as a complete, authoritative answer -- not a diff or
commentary on what changed."""

    if verbatim_hint:
        non_verbatim_block = format_segments_for_verification(verbatim_hint)
        hint_block += f"""

VERBATIM CORRECTION NEEDED: on a first pass, at least one of your extracted segments below was NOT
found as an exact, character-for-character match anywhere in the document text -- meaning it was
paraphrased, summarized, or otherwise regenerated instead of copied. This is a serious problem: it
corrupts the historical record. For EACH segment shown below, locate the ACTUAL passage in the
document text that it was meant to represent, and replace segment_text with that passage copied
EXACTLY -- same characters, same punctuation, same spacing, same script (do not modernize or
translate). If you genuinely cannot find a real corresponding passage for a given segment (i.e. you
cannot locate it verbatim anywhere in the text), drop that segment entirely rather than keep an
invented one.

Segments needing verbatim correction:
{non_verbatim_block}

Return your final, corrected answer using the exact same JSON schema as a complete, authoritative
answer -- not a diff or commentary on what changed."""

    user_prompt = f"""Segment this newspaper document according to the codebook.

Metadata:
DocId: {row["doc_id"]}
Date: {row["date"]}
Title: {row["title"]}
Source: {row["source"]}

Document text (body only -- the Title above is separate; do not require the body to repeat it):
{text}
{hint_block}

{JSON_REQUIREMENTS}
"""

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]

    max_new_tokens = max_new_tokens_override or int(os.environ.get("MAX_TOKENS", "2048"))

    # Qwen3 (including qwen3:30b) auto-enables "thinking" mode in Ollama
    # whenever the `think` field is omitted -- Qwen2.5 doesn't have this
    # concept at all, so this was invisible until the model switch. Left on,
    # a chunk of every num_predict budget goes to an internal <think>...
    # </think> block before the actual JSON, which is a very plausible
    # contributor to truncation on exactly the hardest cases (long documents
    # needing precise extraction) and possibly to answers that don't match
    # their own reasoning. `think` MUST be a top-level request field, not
    # nested inside `options` -- that's a common mistake that silently does
    # nothing. Default off since we want direct structured output, not
    # exposition; override via OLLAMA_THINK=true to inspect the model's
    # reasoning for debugging (added to the debug log when present).
    think_enabled = os.environ.get("OLLAMA_THINK", "false").strip().lower() in ("1", "true", "yes")

    payload = {
        "model": OLLAMA_MODEL,
        "messages": messages,
        "stream": False,
        "think": think_enabled,
        # Ollama's structured-output mode: forces syntactically valid JSON,
        # which should cut down a lot on the parse failures / degenerate
        # outputs seen before. It does not guarantee our *schema* is
        # followed (that's what repair()/normalize_*() are for).
        "format": "json",
        "options": {
            "temperature": float(os.environ.get("TEMPERATURE", "0.0")),
            "num_predict": max_new_tokens,
            "num_ctx": OLLAMA_NUM_CTX,
        },
    }

    print(f"[{now()}] INFERENCE_RUNNING DocId={row['doc_id']} num_ctx={OLLAMA_NUM_CTX} max_new_tokens={max_new_tokens} think={think_enabled}", flush=True)

    try:
        resp = requests.post(f"{OLLAMA_HOST}/api/chat", json=payload, timeout=600)
        resp.raise_for_status()
        data = resp.json()
    except Exception as e:
        # IMPORTANT: this is an infrastructure failure (Ollama unreachable,
        # timed out, bad HTTP response, etc.), NOT a model judgement that
        # nothing is relevant. Using segmentation_label="PROCESSING_ERROR"
        # (not a real label in SEGMENTATION_LABELS) keeps it visibly distinct
        # from a genuine NO_RELEVANT_SEGMENT decision in the output, instead
        # of silently masquerading as one.
        obj = {
            "doc_id": row["doc_id"],
            "segmentation_label": "PROCESSING_ERROR",
            "segmentation_confidence": 0.0,
            "segmentation_explanation": "",
            "segments": [],
        }
        obj["_parse_error"] = f"OLLAMA_REQUEST_FAILED {type(e).__name__}: {e}"
        obj["_raw_output_len"] = 0
        obj["_raw_output_preview"] = ""
        obj["_prompt_tokens"] = None
        obj["_gen_tokens"] = None
        return obj

    text_out = data.get("message", {}).get("content", "")
    thinking_out = data.get("message", {}).get("thinking", "")
    prompt_tokens = data.get("prompt_eval_count")
    gen_tokens = data.get("eval_count")
    print(f"[{now()}] INFERENCE_DONE DocId={row['doc_id']} prompt_tokens={prompt_tokens} gen_tokens={gen_tokens} done_reason={data.get('done_reason')} thinking_chars={len(thinking_out)}", flush=True)

    try:
        obj = extract_json(text_out)
        obj["_parse_error"] = None
    except Exception as e:
        # Same reasoning as above: a JSON parse failure is not a content
        # judgement. Tag it PROCESSING_ERROR, not NO_RELEVANT_SEGMENT.
        obj = {
            "doc_id": row["doc_id"],
            "segmentation_label": "PROCESSING_ERROR",
            "segmentation_confidence": 0.0,
            "segmentation_explanation": "",
            "segments": [],
        }
        obj["_parse_error"] = f"{type(e).__name__}: {e}"

    # Kept only for the debug log (stripped before the doc is finalized), so
    # confidence=0/empty-explanation cases can actually be diagnosed instead
    # of just being silently re-labeled NO_RELEVANT_SEGMENT.
    obj["_raw_output_len"] = len(text_out)
    obj["_raw_output_preview"] = text_out[:4000]
    obj["_thinking_preview"] = thinking_out[:2000] if thinking_out else ""
    obj["_prompt_tokens"] = prompt_tokens
    obj["_gen_tokens"] = gen_tokens
    # done_reason == "length" means Ollama hit num_predict/num_ctx and cut
    # the generation off mid-JSON -- a strong, explicit signal (better than
    # guessing from confidence/explanation alone) that this doc needs a retry
    # with more room.
    obj["_truncated_by_ollama"] = data.get("done_reason") == "length"

    return obj

def is_degenerate(obj: Dict[str, Any]) -> bool:
    """A result that parsed fine but came back with confidence 0 AND no
    explanation at all is much more likely an incomplete/degenerate
    generation than a genuine judgement -- a real NO_RELEVANT_SEGMENT call
    should still be able to say why in one sentence. We use this narrow,
    specific signature (not "confidence is low") so we don't retry every
    legitimate low-confidence NO_RELEVANT_SEGMENT.

    We also retry whenever Ollama itself reports it cut generation short
    (done_reason == "length"), since that's an explicit truncation signal
    rather than a guess, and whenever the doc was tagged PROCESSING_ERROR
    (request failure / unparseable response -- an infra problem, not a
    content judgement)."""
    if obj.get("_parse_error"):
        return True
    if obj.get("segmentation_label") == "PROCESSING_ERROR":
        return True
    if obj.get("_truncated_by_ollama"):
        return True
    try:
        conf = float(obj.get("segmentation_confidence", 0.0) or 0.0)
    except Exception:
        conf = 0.0
    explanation = str(obj.get("segmentation_explanation") or "").strip()
    return conf == 0.0 and not explanation

def call_model_with_retry(row: Dict[str, Any]) -> Dict[str, Any]:
    obj = call_model(row)
    attempts = 0
    base_tokens = int(os.environ.get("MAX_TOKENS", "2048"))
    while attempts < MAX_DEGENERATE_RETRIES and is_degenerate(obj):
        attempts += 1
        # Back off before retrying an actual request failure (connection
        # refused, timeout, 5xx, etc.) -- an immediate retry against a
        # server that's still down/restarting just wastes time and produces
        # the same failure again. No backoff needed for a plain
        # confidence=0/empty-explanation retry, since that's a generation
        # issue on an otherwise-healthy server.
        is_request_failure = str(obj.get("_parse_error", "") or "").startswith("OLLAMA_REQUEST_FAILED")
        if is_request_failure:
            backoff = min(30, 5 * attempts)
            print(f"[{now()}] REQUEST_FAILURE DocId={row['doc_id']} attempt={attempts} backing off {backoff}s before retry", flush=True)
            time.sleep(backoff)
        bumped = base_tokens + attempts * base_tokens  # give it materially more room each retry
        print(f"[{now()}] RETRY_DEGENERATE DocId={row['doc_id']} attempt={attempts} max_new_tokens={bumped}", flush=True)
        obj = call_model(row, max_new_tokens_override=bumped)
    obj["_retries"] = attempts
    return obj

def call_model_with_relevance_recheck(row: Dict[str, Any]) -> Dict[str, Any]:
    """After the normal retry pipeline, if the final answer is
    NO_RELEVANT_SEGMENT but the title/text contains a high-precision
    relevance marker the model apparently missed, give it exactly one more
    targeted attempt with that term called out explicitly. This exists
    because we observed the model flatly denying a term was present when it
    was plainly in the title (e.g. claiming "no identifying terms found"
    when the title literally contained 出洋留學) -- a factual miss, not just
    a borderline judgement call, and worth a second, more focused look
    rather than silently trusting the first answer."""
    obj = call_model_with_retry(row)
    if obj.get("_parse_error") or obj.get("segmentation_label") != "NO_RELEVANT_SEGMENT":
        return obj
    marker = find_relevance_recheck_marker(row.get("title", ""), row.get("text", ""))
    if not marker:
        return obj
    print(f"[{now()}] RELEVANCE_RECHECK DocId={row['doc_id']} matched_marker={marker!r}", flush=True)
    obj2 = call_model(row, relevance_hint=marker)
    if obj2.get("_parse_error"):
        # The recheck call itself failed -- keep the original (already
        # successfully-parsed) NO_RELEVANT_SEGMENT rather than discarding a
        # good result for a failed one.
        return obj
    obj2["_retries"] = obj.get("_retries", 0) + 1
    obj2["_relevance_recheck_marker"] = marker
    return obj2

def call_model_with_verification(row: Dict[str, Any]) -> Dict[str, Any]:
    """Outermost pipeline: after retries + relevance recheck, if
    VERIFY_SEGMENTATION is on and the result is SINGLE_SEGMENT or
    MULTIPLE_SEGMENTS, give the model one more pass to check its own
    boundaries, completeness, and purity against the full document (see the
    VERIFICATION PASS block built in call_model). A verification failure
    (request error / unparseable response) falls back to the already-valid
    first-pass result rather than losing it -- verification can only
    improve or confirm, never destroy, a good first-pass answer."""
    obj = call_model_with_relevance_recheck(row)
    if not VERIFY_SEGMENTATION:
        return obj
    if obj.get("_parse_error") or obj.get("segmentation_label") not in ("SINGLE_SEGMENT", "MULTIPLE_SEGMENTS"):
        return obj

    first_pass_segments = obj.get("segments", [])
    print(f"[{now()}] VERIFICATION_RUNNING DocId={row['doc_id']} first_pass_segments={len(first_pass_segments)}", flush=True)
    verified = call_model(row, verify_segments=first_pass_segments)
    if verified.get("_parse_error"):
        print(f"[{now()}] VERIFICATION_FAILED DocId={row['doc_id']} parse_error={verified.get('_parse_error')!r} -- keeping first-pass result", flush=True)
        return obj

    verified_segments = verified.get("segments", []) if isinstance(verified.get("segments"), list) else []
    changed = (
        len(verified_segments) != len(first_pass_segments)
        or [s.get("segment_text") for s in verified_segments] != [s.get("segment_text") for s in first_pass_segments]
    )
    print(f"[{now()}] {'VERIFICATION_CHANGED' if changed else 'VERIFICATION_CONFIRMED'} DocId={row['doc_id']} "
          f"segments_before={len(first_pass_segments)} segments_after={len(verified_segments)}", flush=True)

    verified["_retries"] = obj.get("_retries", 0) + 1
    verified["_relevance_recheck_marker"] = obj.get("_relevance_recheck_marker")
    verified["_verification_ran"] = True
    verified["_verification_changed"] = changed
    return verified

def call_model_with_verbatim_recheck(row: Dict[str, Any]) -> Dict[str, Any]:
    """Final stage of the pipeline: after retries + relevance recheck +
    optional verification, check whether any extracted segment fails the
    verbatim test (is_verbatim) -- i.e. was paraphrased/regenerated instead
    of copied character-for-character from the source. If so, give the
    model exactly one more targeted pass showing it its own non-verbatim
    segment(s) and asking it to locate and return the real passage instead
    (see the VERBATIM CORRECTION block built in call_model). This mirrors
    the relevance-recheck/verification-pass pattern used elsewhere: a
    second, more targeted look rather than an algorithmic "fix" that risks
    inventing text of its own (e.g. fuzzy string matching against the
    source could easily latch onto the wrong passage). A recheck failure
    (request error / unparseable response) falls back to the first-pass
    result rather than losing it.

    Runs on WHOLE_DOCUMENT results too defensively, though those should
    always be verbatim by construction (segment_text is taken directly from
    the input, never model-generated) -- this only actually fires when
    is_verbatim() finds a genuine mismatch."""
    obj = call_model_with_verification(row)
    if obj.get("_parse_error") or obj.get("segmentation_label") in ("NO_RELEVANT_SEGMENT", "PROCESSING_ERROR"):
        return obj

    segments = obj.get("segments", [])
    if not segments:
        return obj

    original = clean_text(row.get("text", ""))
    non_verbatim = [s for s in segments if not is_verbatim(clean_text(s.get("segment_text", "")), original)]
    if not non_verbatim:
        return obj

    print(f"[{now()}] VERBATIM_RECHECK_RUNNING DocId={row['doc_id']} non_verbatim_segments={len(non_verbatim)}/{len(segments)}", flush=True)
    rechecked = call_model(row, verbatim_hint=non_verbatim)
    if rechecked.get("_parse_error"):
        print(f"[{now()}] VERBATIM_RECHECK_FAILED DocId={row['doc_id']} parse_error={rechecked.get('_parse_error')!r} -- keeping first-pass result", flush=True)
        return obj

    rechecked_segments = rechecked.get("segments", []) if isinstance(rechecked.get("segments"), list) else []
    still_non_verbatim = sum(
        1 for s in rechecked_segments if not is_verbatim(clean_text(s.get("segment_text", "")), original)
    )
    print(f"[{now()}] VERBATIM_RECHECK_{'STILL_BAD' if still_non_verbatim else 'FIXED'} DocId={row['doc_id']} "
          f"non_verbatim_before={len(non_verbatim)} non_verbatim_after={still_non_verbatim} "
          f"segments_before={len(segments)} segments_after={len(rechecked_segments)}", flush=True)

    rechecked["_retries"] = obj.get("_retries", 0) + 1
    rechecked["_relevance_recheck_marker"] = obj.get("_relevance_recheck_marker")
    rechecked["_verification_ran"] = obj.get("_verification_ran", False)
    rechecked["_verification_changed"] = obj.get("_verification_changed", False)
    rechecked["_verbatim_recheck_ran"] = True
    rechecked["_verbatim_recheck_still_bad"] = still_non_verbatim
    return rechecked

def flatten(jsonl_path: Path, csv_path: Path):
    by_id: Dict[str, Dict[str, Any]] = {}
    if jsonl_path.exists():
        for line in jsonl_path.read_text(encoding="utf-8").splitlines():
            if not line.strip():
                continue
            try:
                obj = json.loads(line)
            except Exception:
                continue
            by_id[str(obj.get("doc_id", ""))] = obj

    rows = []
    for obj in by_id.values():
        base = {
            "doc_id": obj.get("doc_id"),
            "date": obj.get("date", ""),
            "title": obj.get("title", ""),
            "source": obj.get("source", ""),
            "segmentation_label": obj.get("segmentation_label"),
            "segmentation_confidence": obj.get("segmentation_confidence", ""),
            "segmentation_explanation": obj.get("segmentation_explanation", ""),
            "processing_error": obj.get("_error", "") or "",
        }

        segs = obj.get("segments", [])
        if not segs:
            r = dict(base)
            r.update({k: "" for k in SEGMENT_FIELDS})
            rows.append(r)
        else:
            for seg in segs:
                r = dict(base)
                r.update({k: seg.get(k, "") for k in SEGMENT_FIELDS})
                rows.append(r)

    pd.DataFrame(rows, columns=CSV_FIELDS).to_csv(csv_path, index=False, quoting=csv.QUOTE_MINIMAL)

def load_done_ids(jsonl_path: Path) -> set:
    done = set()
    if not jsonl_path.exists():
        return done
    for line in jsonl_path.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        try:
            obj = json.loads(line)
        except Exception:
            continue
        if not obj.get("_error"):
            done.add(str(obj.get("doc_id", "")))
    return done

def read_input_csv(path: str) -> pd.DataFrame:
    """The input corpus CSV (an OCR'd historical newspaper export) has, in
    practice, repeatedly turned out to need special handling -- every
    annotation/sample CSV produced from this pipeline so far has needed a
    non-default separator, a UTF-8 BOM-aware encoding, or lenient bad-line
    handling to parse at all. A bare pd.read_csv(path) is fragile against
    exactly this: a single stray unescaped delimiter or quote character
    inside an OCR'd Title/Text field is enough to make pandas' C parser
    raise "Error tokenizing data" and abort the entire run before a single
    document is processed.

    IMPORTANT: a strategy "succeeding" (raising no exception) is NOT enough
    to trust it -- e.g. reading a semicolon-delimited file with the default
    comma separator raises no error, it just silently produces one giant
    column instead of the real ones; a BOM can silently corrupt the first
    column's name; a stray unescaped delimiter inside a text field can
    silently shift every value in that row one column to the left. All of
    these are worse than a crash, since they'd let the run proceed on
    corrupted/misaligned data without anyone noticing. So every attempt is
    validated (expected columns present, DocId not entirely empty) before
    being accepted, not just checked for exceptions.
    """
    required_cols = {"DocId", "Date", "Title", "Source", "Text"}
    attempts = [
        ("default (comma, strict)", {"index_col": False}),
        ("utf-8-sig encoding (BOM)", {"index_col": False, "encoding": "utf-8-sig"}),
        ("auto-detect delimiter", {"index_col": False, "sep": None, "engine": "python"}),
        ("utf-8-sig + auto-detect delimiter", {"index_col": False, "sep": None, "engine": "python", "encoding": "utf-8-sig"}),
        ("lenient: skip malformed rows", {"index_col": False, "engine": "python", "on_bad_lines": "warn"}),
        ("lenient + utf-8-sig: skip malformed rows", {"index_col": False, "engine": "python", "on_bad_lines": "warn", "encoding": "utf-8-sig"}),
    ]
    # index_col=False on every attempt: without it, pandas can silently treat
    # a row with MORE fields than the header as having an implicit leading
    # index column instead of erroring -- which is exactly how an unescaped
    # delimiter inside one text field can shift DocId/Date/Title/Source
    # values by one column with no warning at all. Confirmed by testing.
    last_err = None
    for i, (label, kwargs) in enumerate(attempts):
        try:
            df = pd.read_csv(path, **kwargs)
        except Exception as e:
            last_err = e
            print(f"[{now()}] INPUT_CSV_READ attempt {i} ({label}) raised: {type(e).__name__}: {e}", flush=True)
            continue

        missing = required_cols - set(df.columns)
        if missing:
            preview = list(df.columns)[:8]
            print(f"[{now()}] INPUT_CSV_READ attempt {i} ({label}) parsed without error but is "
                  f"missing expected column(s) {missing} -- got columns {preview}{'...' if len(df.columns) > 8 else ''}. "
                  f"This usually means the wrong delimiter or encoding was used. Trying next strategy.", flush=True)
            last_err = last_err or ValueError(f"parsed but missing columns {missing}")
            continue
        if df["DocId"].isna().all():
            print(f"[{now()}] INPUT_CSV_READ attempt {i} ({label}) parsed with the right columns but "
                  f"every DocId is empty/NaN -- likely a column-alignment problem (e.g. an unescaped "
                  f"delimiter inside a text field shifting values). Trying next strategy.", flush=True)
            last_err = last_err or ValueError("parsed but DocId entirely empty")
            continue

        # A non-empty DocId column isn't enough: the classic symptom of an
        # unescaped delimiter inside a text field shifting every value one
        # column over is that DocId silently ends up holding what should be
        # the Date column's values (or similar) -- no NaNs anywhere, just
        # wrong. Sanity-check on a sample rather than trust a merely
        # non-empty column.
        sample = df.head(200)
        docid_eq_date = (sample["DocId"].astype(str) == sample["Date"].astype(str)).mean()
        docid_uniqueness = sample["DocId"].nunique() / max(len(sample), 1)
        if docid_eq_date > 0.5 or docid_uniqueness < 0.3:
            print(f"[{now()}] INPUT_CSV_READ attempt {i} ({label}) parsed with the right column names, "
                  f"but DocId looks wrong (matches Date in {docid_eq_date:.0%} of a sample, only "
                  f"{docid_uniqueness:.0%} unique) -- likely column misalignment from an unescaped "
                  f"delimiter inside a text field. Trying next strategy.", flush=True)
            last_err = last_err or ValueError("DocId column looks misaligned (matches Date / low uniqueness)")
            continue

        if i > 0:
            print(f"[{now()}] INPUT_CSV_READ default parsing did not produce valid data; succeeded "
                  f"with fallback strategy #{i} ({label}). Consider re-saving the input file in a more "
                  f"standard format if this keeps happening, but it's safe to proceed as-is.", flush=True)
        return df

    raise SystemExit(
        f"Could not read input CSV {path} into a valid DataFrame (with non-empty DocId/Date/Title/"
        f"Source/Text columns) with any known strategy. Last error: {last_err}\n"
        f"This usually means a Title/Text field contains an unescaped delimiter or quote character "
        f"that breaks column alignment even in lenient mode, and no automatic strategy can safely "
        f"guess the correct alignment. Try: (1) opening the file in a spreadsheet app and re-saving "
        f"as a clean UTF-8 CSV with proper quoting, (2) checking whether the real delimiter is ';' or "
        f"a tab rather than ',', (3) grep-ing the Text column for stray unescaped commas or unmatched "
        f"double-quote characters."
    )

def main():
    print(f"[{now()}] PATCH_VERSION={PATCH_VERSION}", flush=True)
    print(f"[{now()}] OLLAMA_HOST={OLLAMA_HOST} OLLAMA_MODEL={OLLAMA_MODEL} OLLAMA_NUM_CTX={OLLAMA_NUM_CTX}", flush=True)
    print(f"[{now()}] PROMPT_VARIANT={PROMPT_VARIANT} (system_prompt_chars={len(SYSTEM_PROMPT)})", flush=True)
    print(f"[{now()}] OLLAMA_THINK={os.environ.get('OLLAMA_THINK', 'false')} (qwen3 thinking mode; 'false' means direct JSON output, no wasted reasoning-token budget)", flush=True)
    print(f"[{now()}] VERIFY_SEGMENTATION={VERIFY_SEGMENTATION} (second pass on SINGLE_SEGMENT/MULTIPLE_SEGMENTS results; roughly doubles cost for those docs)", flush=True)
    if not wait_for_ollama(timeout=60):
        raise SystemExit(
            f"Could not reach Ollama at {OLLAMA_HOST}/api/tags after 60s. "
            f"Check that `ollama serve` is actually running (on this infra it's usually "
            f"already up -- `ollama list` should work in a terminal on the same node), "
            f"and that OLLAMA_HOST points at the right host:port."
        )

    ap = argparse.ArgumentParser()
    ap.add_argument("--input", required=True)
    ap.add_argument("--output-dir", required=True)
    ap.add_argument("--batch", type=int, default=None)
    args = ap.parse_args()

    outdir = Path(args.output_dir)
    outdir.mkdir(parents=True, exist_ok=True)
    jsonl_path = outdir / f"segments_{PROMPT_VARIANT}.jsonl"
    csv_path = outdir / f"segments_{PROMPT_VARIANT}.csv"
    debug_log_path = outdir / f"segments_debug_{PROMPT_VARIANT}.jsonl"

    # One-time migration: earlier versions of this script (before the
    # full/trimmed prompt A/B option existed) wrote to unsuffixed filenames.
    # If those exist and the new variant-suffixed "full" files don't yet,
    # adopt them as-is -- this is a faithful rename, not a new run, since
    # "full" is what every prior run actually used. Without this, switching
    # to variant-suffixed filenames would look like 0 docs done and silently
    # reprocess an entire already-completed corpus.
    if PROMPT_VARIANT == "full":
        legacy_jsonl = outdir / "segments.jsonl"
        legacy_csv = outdir / "segments.csv"
        legacy_debug = outdir / "segments_debug.jsonl"
        if legacy_jsonl.exists() and not jsonl_path.exists():
            print(f"[{now()}] MIGRATING legacy {legacy_jsonl} -> {jsonl_path} (one-time; preserves prior progress under the 'full' variant)", flush=True)
            legacy_jsonl.rename(jsonl_path)
            if legacy_csv.exists() and not csv_path.exists():
                legacy_csv.rename(csv_path)
            if legacy_debug.exists() and not debug_log_path.exists():
                legacy_debug.rename(debug_log_path)

    df = read_input_csv(args.input)

    todo = []
    for _, r in df.iterrows():
        doc_id = str(r.get("DocId", ""))
        if not doc_id:
            continue
        title = clean_text(r.get("Title", ""))
        body = clean_text(r.get("Text", ""))
        # NOTE: title is deliberately NOT concatenated into "text" here.
        # segment_text/original must stay pure body text -- the title is
        # passed to the model separately as metadata (see the "Title:" line
        # in call_model's user_prompt) and the system prompt instructs the
        # model to weigh it equally, without it ever being written into the
        # document text itself or leaking into segment_text.
        todo.append({
            "doc_id": doc_id,
            "date": str(r.get("Date", "")),
            "title": title,
            "source": str(r.get("Source", "")),
            "text": body,
        })

    # Resume: skip docs already completed (without _error) in the cumulative JSONL,
    # then take only the next BATCH_SIZE undone docs for THIS run.
    done_ids = load_done_ids(jsonl_path)
    remaining = [row for row in todo if row["doc_id"] not in done_ids]
    batch = args.batch
    this_run = remaining if not batch else remaining[:batch]
    print(f"[{now()}] TOTAL={len(todo)} ALREADY_DONE={len(done_ids)} REMAINING={len(remaining)} THIS_RUN={len(this_run)} CHECKPOINT_EVERY={CHECKPOINT_EVERY}", flush=True)

    completed = 0
    degenerate_after_retry = 0
    consecutive_errors = 0
    with jsonl_path.open("a", encoding="utf-8") as f, debug_log_path.open("a", encoding="utf-8") as dbg:
        for i, row in enumerate(this_run, start=1):
            print(f"[{now()}] START {i}/{len(this_run)} DocId={row['doc_id']}", flush=True)
            try:
                obj = call_model_with_verbatim_recheck(row)
                still_degenerate = is_degenerate(obj)
                parse_error_text = obj.get("_parse_error")
                relevance_recheck_marker = obj.get("_relevance_recheck_marker")
                verification_ran = obj.get("_verification_ran", False)
                verification_changed = obj.get("_verification_changed", False)
                verbatim_recheck_ran = obj.get("_verbatim_recheck_ran", False)
                verbatim_recheck_still_bad = obj.get("_verbatim_recheck_still_bad", 0)
                # Always keep a debug trail for anything that needed a retry
                # or is still degenerate after retrying, so failures can be
                # inspected (raw model output, token counts) instead of just
                # showing up as an unexplained confidence=0 row.
                if obj.get("_retries") or still_degenerate or verification_ran or verbatim_recheck_ran:
                    dbg.write(json.dumps({
                        "doc_id": row["doc_id"],
                        "still_degenerate": still_degenerate,
                        "retries": obj.get("_retries", 0),
                        "parse_error": parse_error_text,
                        "relevance_recheck_marker": relevance_recheck_marker,
                        "verification_ran": verification_ran,
                        "verification_changed": verification_changed,
                        "verbatim_recheck_ran": verbatim_recheck_ran,
                        "verbatim_recheck_still_bad": verbatim_recheck_still_bad,
                        "prompt_tokens": obj.get("_prompt_tokens"),
                        "gen_tokens": obj.get("_gen_tokens"),
                        "raw_output_len": obj.get("_raw_output_len"),
                        "raw_output_preview": obj.get("_raw_output_preview"),
                        "thinking_preview": obj.get("_thinking_preview"),
                        "truncated_by_ollama": obj.get("_truncated_by_ollama"),
                        "input_text_len": len(row["text"]),
                    }, ensure_ascii=False) + "\n")
                    dbg.flush()
                    if still_degenerate:
                        degenerate_after_retry += 1
                for k in ("_parse_error", "_raw_output_len", "_raw_output_preview", "_thinking_preview", "_prompt_tokens", "_gen_tokens", "_retries", "_truncated_by_ollama", "_relevance_recheck_marker", "_verification_ran", "_verification_changed", "_verbatim_recheck_ran", "_verbatim_recheck_still_bad"):
                    obj.pop(k, None)
                if still_degenerate:
                    # Persist the failure so load_done_ids() excludes this doc
                    # from "done" -- it will be automatically retried the next
                    # time this script runs, instead of being stuck forever
                    # looking like a legitimate (but unexplained) judgement.
                    obj["_error"] = parse_error_text or "degenerate after retries (confidence=0, empty explanation)"
                    consecutive_errors += 1
                else:
                    consecutive_errors = 0
                obj = repair(obj, row)
            except Exception as e:
                obj = {
                    "doc_id": row["doc_id"],
                    "segmentation_label": "PROCESSING_ERROR",
                    "segmentation_confidence": 0.0,
                    "segmentation_explanation": "",
                    "segments": [],
                    "_error": f"ERROR: {type(e).__name__}: {e}",
                }
                consecutive_errors += 1

            # carry metadata for the CSV
            obj["date"] = row["date"]
            obj["title"] = row["title"]
            obj["source"] = row["source"]

            f.write(json.dumps(obj, ensure_ascii=False) + "\n")
            f.flush()
            os.fsync(f.fileno())
            completed += 1

            print(f"[{now()}] DONE {i}/{len(this_run)} DocId={row['doc_id']} seg={obj.get('segmentation_label')}", flush=True)

            if CHECKPOINT_EVERY > 0 and completed % CHECKPOINT_EVERY == 0:
                flatten(jsonl_path, csv_path)
                print(f"[{now()}] CHECKPOINT_FLUSH completed={completed} -> {csv_path}", flush=True)

            # Circuit breaker: a long run of consecutive failures (like the
            # 100-in-a-row incident this was written to catch) almost always
            # means the infra is down, not that 100 documents in a row are
            # genuinely irrelevant. Pause and re-check health instead of
            # silently grinding through the rest of the corpus writing
            # error placeholders.
            if consecutive_errors >= CONSECUTIVE_ERROR_PAUSE_THRESHOLD:
                print(f"[{now()}] CIRCUIT_BREAKER {consecutive_errors} consecutive failures -- pausing to check Ollama health", flush=True)
                if wait_for_ollama(timeout=300):
                    print(f"[{now()}] CIRCUIT_BREAKER Ollama reachable again, resuming", flush=True)
                    consecutive_errors = 0
                else:
                    print(f"[{now()}] CIRCUIT_BREAKER Ollama still unreachable after an extended wait -- "
                          f"stopping this run early instead of burning through the rest of the corpus as "
                          f"error placeholders. Already-completed docs are safe (checkpointed), and every "
                          f"failed doc in this run was tagged with an error, so it will be retried "
                          f"automatically the next time this script runs. Fix Ollama, then rerun.", flush=True)
                    break

    flatten(jsonl_path, csv_path)
    done_total = len(done_ids) + completed
    print(f"[{now()}] WROTE_JSONL={jsonl_path}", flush=True)
    print(f"[{now()}] WROTE_CSV={csv_path}", flush=True)
    print(f"[{now()}] WROTE_DEBUG_LOG={debug_log_path}", flush=True)
    print(f"[{now()}] BATCH_DONE completed_this_run={completed} done_total={done_total} remaining_after={len(todo) - done_total} degenerate_after_retry={degenerate_after_retry}", flush=True)

if __name__ == "__main__":
    main()
'''


OLLAMA_HOST resolved to http://127.0.0.1:65330 (source: module load ollama (login shell))


In [2]:
# Local execution: we're already on an allocated compute node via JupyterHub,
# so there is no SSH/sbatch step here -- we just write the worker script to
# disk and run it as a subprocess, streaming its stdout back into the notebook.

if CLEAN_LOCAL_OUTPUT_DIR and LOCAL_OUTPUT_DIR.exists():
    import shutil
    shutil.rmtree(LOCAL_OUTPUT_DIR)

LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
WORKER_SCRIPT_PATH.write_text(SEGMENT_ARTICLES_PY, encoding="utf-8")
print(f"Wrote worker script -> {WORKER_SCRIPT_PATH.resolve()}")

# Quick sanity check before kicking off the full batch: confirm the input
# CSV exists and Ollama is actually reachable, so failures show up here
# instead of 100 docs into a run.
assert Path(LOCAL_INPUT_CSV).exists(), f"Input CSV not found: {LOCAL_INPUT_CSV}"

try:
    import requests
    r = requests.get(f"{OLLAMA_HOST}/api/tags", timeout=5)
    r.raise_for_status()
    names = [m.get("name") for m in r.json().get("models", [])]
    print(f"Ollama reachable at {OLLAMA_HOST}. Models available: {names}")
    if OLLAMA_MODEL not in names:
        print(f"WARNING: '{OLLAMA_MODEL}' not found in that list -- check the exact tag with `ollama list`.")
except Exception as e:
    print(f"WARNING: could not reach Ollama at {OLLAMA_HOST}/api/tags ({e}). "
          f"Run `ollama list` in a terminal on this same node to confirm the server is up.")

env = os.environ.copy()
env.update({
    "OLLAMA_HOST": OLLAMA_HOST,
    "OLLAMA_MODEL": OLLAMA_MODEL,
    "OLLAMA_NUM_CTX": str(OLLAMA_NUM_CTX),
    "OLLAMA_THINK": str(OLLAMA_THINK).lower(),
    "VERIFY_SEGMENTATION": str(VERIFY_SEGMENTATION).lower(),
    "MAX_TOKENS": str(MAX_TOKENS),
    "TEMPERATURE": str(TEMPERATURE),
    "MAX_CHARS": str(MAX_CHARS),
    "CHECKPOINT_EVERY": str(CHECKPOINT_EVERY),
    "MAX_DEGENERATE_RETRIES": str(MAX_DEGENERATE_RETRIES),
    "PROMPT_VARIANT": PROMPT_VARIANT,
})

cmd = [
    sys.executable, str(WORKER_SCRIPT_PATH),
    "--input", str(LOCAL_INPUT_CSV),
    "--output-dir", str(LOCAL_OUTPUT_DIR),
]
if BATCH_SIZE:
    cmd += ["--batch", str(BATCH_SIZE)]

print(f"PROMPT_VARIANT={PROMPT_VARIANT} -> output files: {_variant_paths(PROMPT_VARIANT)['csv'].name}, "
      f"{_variant_paths(PROMPT_VARIANT)['jsonl'].name}")
print("Running:", " ".join(cmd), f"(background={RUN_IN_BACKGROUND})")

if RUN_IN_BACKGROUND:
    log_f = open(LOG_PATH, "a", buffering=1)
    proc = subprocess.Popen(
        cmd, env=env, stdout=log_f, stderr=subprocess.STDOUT,
        stdin=subprocess.DEVNULL, start_new_session=True,  # detach from this kernel
    )
    print(f"Started in background. PID={proc.pid}  log={LOG_PATH.resolve()}")
    print("This keeps running even if the browser tab closes or the kernel restarts.")
    print("Use the log-tailing cell (next) to watch progress, and the status cell to spot-check output.")
    print(f"To stop it manually: `kill {proc.pid}`  or  `pkill -f {WORKER_SCRIPT_PATH.name}`")
else:
    proc = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    print(f"\nWorker process exited with code {proc.returncode}")


Wrote worker script -> /sps/humanum/home/carmand01/test/segment_articles.py
Ollama reachable at http://127.0.0.1:65330. Models available: ['qwen3.5:35b', 'llama3.3:70b', 'qwen3:30b', 'qwen2.5:7b', 'mxbai-embed-large:latest', 'llama3.2:3b', 'nomic-embed-text:latest']
PROMPT_VARIANT=full -> output files: segments_full.csv, segments_full.jsonl
Running: /pbs/software/redhat-9-x86_64/jnp/venvs/py3.13.11/bin/python segment_articles.py --input ./data/shenbao_clean.csv --output-dir data --batch 1000 (background=True)
Started in background. PID=2254749  log=/sps/humanum/home/carmand01/test/data/run_full.log
This keeps running even if the browser tab closes or the kernel restarts.
Use the log-tailing cell (next) to watch progress, and the status cell to spot-check output.
To stop it manually: `kill 2254749`  or  `pkill -f segment_articles.py`


In [ ]:
# Safe to run/interrupt any time -- this only WATCHES the log file; interrupting
# this cell (Kernel -> Interrupt, or stopping the cell) does NOT stop the
# background job, it just stops tailing it.
import time

if not LOG_PATH.exists():
    print(f"No log yet at {LOG_PATH.resolve()} -- has the run cell above been executed?")
else:
    print(f"Tailing {LOG_PATH.resolve()} (interrupt this cell to stop watching; the job keeps running)\n")
    with open(LOG_PATH, "r", encoding="utf-8") as f:
        f.seek(0, 2)  # start from the end
        try:
            while True:
                line = f.readline()
                if line:
                    print(line, end="")
                else:
                    time.sleep(2)
        except KeyboardInterrupt:
            print("\n[stopped watching -- background job is still running]")


Tailing /sps/humanum/home/carmand01/test/data/run_full.log (interrupt this cell to stop watching; the job keeps running)

[2026-09-09 15:51:00] PATCH_VERSION=batch_resume_v15_verbatim_recheck_relevance_evidence
[2026-09-09 15:51:00] OLLAMA_HOST=http://127.0.0.1:65330 OLLAMA_MODEL=qwen3.5:35b OLLAMA_NUM_CTX=32768
[2026-09-09 15:51:00] PROMPT_VARIANT=full (system_prompt_chars=23556)
[2026-09-09 15:51:00] OLLAMA_THINK=false (qwen3 thinking mode; 'false' means direct JSON output, no wasted reasoning-token budget)
[2026-09-09 15:51:00] VERIFY_SEGMENTATION=False (second pass on SINGLE_SEGMENT/MULTIPLE_SEGMENTS results; roughly doubles cost for those docs)
[2026-09-09 15:51:00] OLLAMA_REACHABLE host=http://127.0.0.1:65330 models=['qwen3.5:35b', 'llama3.3:70b', 'qwen3:30b', 'qwen2.5:7b', 'mxbai-embed-large:latest', 'llama3.2:3b', 'nomic-embed-text:latest']
[2026-09-09 15:51:04] TOTAL=23974 ALREADY_DONE=16619 REMAINING=7374 THIS_RUN=1000 CHECKPOINT_EVERY=10
[2026-09-09 15:51:04] START 1/1000 Do

In [ ]:
# Quick progress + quality spot-check. Safe to re-run any time, including
# while the background job is still going (CHECKPOINT_EVERY flushes the CSV
# periodically, and the JSONL is append-only). Reports on PROMPT_VARIANT's
# output files -- switch that variable and rerun this cell to inspect the
# other variant's results.
import json
import pandas as pd

_paths = _variant_paths(PROMPT_VARIANT)
jsonl_path, csv_path, debug_path = _paths["jsonl"], _paths["csv"], _paths["debug"]
print(f"Reporting on PROMPT_VARIANT={PROMPT_VARIANT!r}: {csv_path.name}")

total_docs = None
try:
    total_docs = sum(1 for _ in open(LOCAL_INPUT_CSV, encoding="utf-8")) - 1  # minus header
except Exception:
    pass

n_done = sum(1 for _ in open(jsonl_path, encoding="utf-8")) if jsonl_path.exists() else 0
pct = f" ({100 * n_done / total_docs:.1f}%)" if total_docs else ""
print(f"Docs completed: {n_done}" + (f" / {total_docs}{pct}" if total_docs else ""))

if debug_path.exists():
    n_debug = sum(1 for _ in open(debug_path, encoding="utf-8"))
    print(f"Docs that needed a retry or are still flagged degenerate (see {debug_path.name}): {n_debug}")
    try:
        n_verbatim_recheck = 0
        n_verbatim_still_bad = 0
        for line in open(debug_path, encoding="utf-8"):
            rec = json.loads(line)
            if rec.get("verbatim_recheck_ran"):
                n_verbatim_recheck += 1
                if rec.get("verbatim_recheck_still_bad"):
                    n_verbatim_still_bad += 1
        if n_verbatim_recheck:
            print(f"Docs that needed a verbatim recheck: {n_verbatim_recheck} "
                  f"(still non-verbatim even after the recheck: {n_verbatim_still_bad})")
    except Exception:
        pass

if csv_path.exists():
    df = pd.read_csv(csv_path)
    print(f"\nRows in flattened CSV: {len(df)}")
    print("\nsegmentation_label counts:")
    print(df["segmentation_label"].value_counts(dropna=False))
    relevant = df[df["segmentation_label"] != "NO_RELEVANT_SEGMENT"]
    print("\ntopic_type counts (segment-level rows only):")
    print(relevant["topic_type"].value_counts(dropna=False))
    print("\nrelevance_to_original_topic counts:")
    print(relevant["relevance_to_original_topic"].value_counts(dropna=False))
    if "relevance_evidence" in relevant.columns:
        n_us_no_evidence = ((relevant["relevance_to_original_topic"] == "RELEVANT_US") & (relevant["relevance_evidence"].fillna("") == "")).sum()
        if n_us_no_evidence:
            print(f"WARNING: {n_us_no_evidence} RELEVANT_US row(s) have no relevance_evidence -- "
                  f"this shouldn't happen (the label is derived FROM the evidence check), worth a look.")
    print("\ngender_mentioned counts:")
    print(relevant["gender_mentioned"].value_counts(dropna=False))
    if "segment_verbatim" in relevant.columns:
        n_non_verbatim = (relevant["segment_verbatim"] == False).sum()
        if n_non_verbatim:
            print(f"\nWARNING: {n_non_verbatim} relevant segment(s) flagged segment_verbatim=False -- "
                  f"the model likely paraphrased/regenerated the text instead of copying it. Worth a "
                  f"look: relevant[relevant.segment_verbatim == False]")
        else:
            print("\nsegment_verbatim: all relevant segments copied verbatim from the source. Good.")
    print("\nSample of relevant rows:")
    display(relevant.sample(min(5, len(relevant))) if len(relevant) else relevant)
else:
    print("No CSV yet -- has at least one CHECKPOINT_FLUSH happened in the log?")


Reporting on PROMPT_VARIANT='full': segments_full.csv


In [ ]:
# ONE-TIME CLEANUP for rows written before this fix (see PATCH_VERSION history):
# a bug in call_model()'s error handling meant genuine infra failures (Ollama
# request errors, unparseable responses) were silently written to the JSONL
# looking exactly like a real "NO_RELEVANT_SEGMENT, confidence=0, no
# explanation" judgement -- with no error marker, so they were never queued
# for automatic retry either. This purges any row matching that exact
# degenerate signature so load_done_ids() will pick them back up and
# call_model_with_retry() will reprocess them for real (now correctly tagged
# PROCESSING_ERROR + auto-retried if it happens again, with the circuit
# breaker protecting against another silent 100-in-a-row incident).
#
# Safe to run multiple times; makes a timestamped backup before touching
# anything, and only removes rows matching the exact stale signature (a
# genuine, correctly-labeled NO_RELEVANT_SEGMENT with a real explanation is
# left untouched).
import json
import shutil
import time as _time

# Reuse the real flatten() from the worker script itself (rather than
# reimplementing CSV-writing logic here and risking it drifting out of sync).
_worker_ns = {"__name__": "_worker_helpers"}
exec(compile(SEGMENT_ARTICLES_PY, "segment_articles.py", "exec"), _worker_ns)
flatten = _worker_ns["flatten"]

jsonl_path = _variant_paths(PROMPT_VARIANT)["jsonl"]
csv_path = _variant_paths(PROMPT_VARIANT)["csv"]
print(f"Cleaning PROMPT_VARIANT={PROMPT_VARIANT!r}: {jsonl_path.name}")

if not jsonl_path.exists():
    print(f"No {jsonl_path} yet -- nothing to clean.")
else:
    backup_path = jsonl_path.with_suffix(f".jsonl.bak.{int(_time.time())}")
    shutil.copy2(jsonl_path, backup_path)
    print(f"Backed up -> {backup_path}")

    kept, purged = [], []
    for line in jsonl_path.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        obj = json.loads(line)
        is_stale_degenerate = (
            obj.get("segmentation_label") == "NO_RELEVANT_SEGMENT"
            and float(obj.get("segmentation_confidence", 0) or 0) == 0.0
            and not str(obj.get("segmentation_explanation") or "").strip()
            and not obj.get("_error")  # already-tagged failures are handled fine as-is
        )
        (purged if is_stale_degenerate else kept).append((obj, line))

    with jsonl_path.open("w", encoding="utf-8") as f:
        for _, line in kept:
            f.write(line + "\n")

    print(f"Kept {len(kept)} rows, purged {len(purged)} stale degenerate rows.")
    if purged:
        print("Purged doc_ids:", [o["doc_id"] for o, _ in purged])
    print("\nRe-flattening CSV to match...")
    flatten(jsonl_path, csv_path)
    print("Done. Rerun the run cell (cell 2) -- purged docs are no longer in "
          "done_ids and will be reprocessed automatically.")


In [ ]:
# CONDITION COMPARISON: runs the SAME small, fixed sample of documents
# through each named condition below (each can vary OLLAMA_MODEL and/or
# PROMPT_VARIANT independently), in isolated subdirectories so this never
# interferes with a full-corpus run already going under LOCAL_OUTPUT_DIR.
# Runs in the foreground (blocking) by design -- this is a deliberate
# small-sample test, not the big batch.
#
# Default below answers "qwen3:30b vs qwen2.5:7b, same (full) prompt" --
# edit CONDITIONS to compare whatever you actually want to know, e.g. add a
# third entry for qwen3:30b + trimmed prompt, or any other model you've
# pulled with `ollama pull`.
#
# The sample is the first COMPARISON_SAMPLE_SIZE rows of your input CSV --
# every condition gets the exact same docs, since each starts from its own
# empty subdirectory. ~30-50 is usually enough to see whether conditions
# meaningfully disagree; increase for a more reliable signal at the cost of
# more time.
#
# COMPARISON_DIR is wiped at the start of every run of this cell (stale
# results from a previous notebook version would silently compare against
# script/prompt content that no longer exists). Set CLEAN_COMPARISON_DIR =
# False to resume a specific in-progress comparison instead.
import shutil as _shutil
import subprocess as _subprocess
import pandas as pd

CONDITIONS = {
    "qwen3_30b":   {"OLLAMA_MODEL": "qwen3:30b",  "PROMPT_VARIANT": "full"},
    "qwen25_7b":   {"OLLAMA_MODEL": "qwen2.5:7b", "PROMPT_VARIANT": "full"},
    # "qwen3_30b_trimmed": {"OLLAMA_MODEL": "qwen3:30b", "PROMPT_VARIANT": "trimmed"},
}

COMPARISON_SAMPLE_SIZE = 40
CLEAN_COMPARISON_DIR = True
COMPARISON_DIR = LOCAL_OUTPUT_DIR / "condition_comparison"

if CLEAN_COMPARISON_DIR and COMPARISON_DIR.exists():
    _shutil.rmtree(COMPARISON_DIR)
COMPARISON_DIR.mkdir(parents=True, exist_ok=True)

WORKER_SCRIPT_PATH.write_text(SEGMENT_ARTICLES_PY, encoding="utf-8")

condition_dfs = {}
for name, overrides in CONDITIONS.items():
    out_dir = COMPARISON_DIR / name
    out_dir.mkdir(parents=True, exist_ok=True)

    env = os.environ.copy()
    env.update({
        "OLLAMA_HOST": OLLAMA_HOST,
        "OLLAMA_MODEL": overrides.get("OLLAMA_MODEL", OLLAMA_MODEL),
        "OLLAMA_NUM_CTX": str(overrides.get("OLLAMA_NUM_CTX", OLLAMA_NUM_CTX)),
        "OLLAMA_THINK": str(overrides.get("OLLAMA_THINK", OLLAMA_THINK)).lower(),
        "VERIFY_SEGMENTATION": str(overrides.get("VERIFY_SEGMENTATION", VERIFY_SEGMENTATION)).lower(),
        "MAX_TOKENS": str(overrides.get("MAX_TOKENS", MAX_TOKENS)),
        "TEMPERATURE": str(overrides.get("TEMPERATURE", TEMPERATURE)),
        "MAX_CHARS": str(overrides.get("MAX_CHARS", MAX_CHARS)),
        "CHECKPOINT_EVERY": str(CHECKPOINT_EVERY),
        "MAX_DEGENERATE_RETRIES": str(MAX_DEGENERATE_RETRIES),
        "PROMPT_VARIANT": overrides.get("PROMPT_VARIANT", "full"),
    })
    cmd = [
        sys.executable, str(WORKER_SCRIPT_PATH),
        "--input", str(LOCAL_INPUT_CSV),
        "--output-dir", str(out_dir),
        "--batch", str(COMPARISON_SAMPLE_SIZE),
    ]
    print(f"\n=== Running condition={name!r} ({overrides}) on {COMPARISON_SAMPLE_SIZE} docs ===")
    proc = _subprocess.Popen(cmd, env=env, stdout=_subprocess.PIPE, stderr=_subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    proc.wait()

    variant = overrides.get("PROMPT_VARIANT", "full")
    csv_path = out_dir / f"segments_{variant}.csv"
    condition_dfs[name] = pd.read_csv(csv_path) if csv_path.exists() else None

# --- Compare ---
missing = [name for name, df in condition_dfs.items() if df is None]
if missing:
    print(f"\nThese conditions produced no output -- check the logs above for errors: {missing}")

available = {name: df for name, df in condition_dfs.items() if df is not None}
if len(available) < 2:
    print("\nNeed at least 2 successful conditions to compare.")
else:
    labels_by_condition = {
        name: df.drop_duplicates("doc_id").set_index("doc_id")["segmentation_label"]
        for name, df in available.items()
    }
    common_ids = None
    for s in labels_by_condition.values():
        common_ids = s.index if common_ids is None else common_ids.intersection(s.index)

    print(f"\n{'='*60}")
    print(f"Compared {len(common_ids)} docs present in all {len(available)} conditions.")

    for name, s in labels_by_condition.items():
        print(f"\n{name!r} segmentation_label counts:")
        print(s.loc[common_ids].value_counts())

    side_by_side = pd.DataFrame({name: s.loc[common_ids] for name, s in labels_by_condition.items()})
    all_agree = side_by_side.apply(lambda row: row.nunique() == 1, axis=1)
    print(f"\nAll-conditions agreement on segmentation_label: {all_agree.mean()*100:.1f}% "
          f"({all_agree.sum()}/{len(all_agree)})")

    disagreements = side_by_side[~all_agree]
    print(f"\n{len(disagreements)} docs where conditions disagree -- these are exactly the ones worth "
          f"manually checking against your own judgement:")
    display(disagreements)

    print(f"\nFull per-segment results saved under: {COMPARISON_DIR}")
    for name in available:
        variant = CONDITIONS[name].get("PROMPT_VARIANT", "full")
        print(f"  {name}: {COMPARISON_DIR / name / f'segments_{variant}.csv'}")
    print("\nNext step: annotate a sample of these (same process as before) to see which condition is "
          "actually more accurate, not just which is faster/cheaper. Agreement rate alone doesn't tell "
          "you which one is RIGHT when they disagree -- especially since none of your annotated samples "
          "so far were run with qwen2.5:7b, so you don't yet have ground truth to compare either model "
          "against on the SAME documents.")
